# ARC Prize 2026 — H3: Program Synthesis + Verified-Set Disambiguation (Qwen3.8-27B)

**Hypothesis (see HYPOTHESIS.md in the repo):** Qwen3.8-27B's coding is frontier-class
(SWE-bench Pro 61.7 > Opus4.6 Max 53.4) while its abstract reasoning is not (HLE 30.8 vs 40.0).
So we never ask it for answer grids — we ask it for `def transform(grid)` programs, verify them
exactly against the train pairs (a free deterministic oracle), and resolve the **disagreement set**
(verified programs that disagree on the test input) with a principled, zero-token layer:

- **S1 symmetry equivariance** — D4 + data-plausible color-permutation pseudo-labels, two-stage gated
  (data consistency, then pool-majority resolvability), train-side falsification + test-side self-consistency
- **S1c unanimous shape-law check** — programs contradicting every train-consistent shape law are penalized
- **S2 LOO corroboration** (gated) — held-out-fold resynthesis; passing a withheld pair is direct generalization evidence
- **S3 MDL prior + S3b literal-memorization screen** — AST size; embedded train grids are disqualifying
- **S4 behavioral clustering** on perturbed test inputs — independent-agreement support
- **S5 soft fallback** — verification failure never discards (Land: precision ~97%, recall ~43%)
- **S6 determinism check**, **S7 adjudication call** (gated), **pass@2 shape-diverse selection**

**Attachments required:** competition data (`arc-prize-2026-arc-agi-2`) and a Qwen3.8-27B FP8 model.
Attach any FP8 mirror in the Kaggle UI (Add Input → Models); the runner auto-detects the mount path
from candidates including `qwen3-8-27b-fp8` (michaelpoluektov / hark99), `qwen3-8-27b-fp8-official`
(mikedan7), `qwen-qwen3-8-27b-fp8` (dangkhoa2016). Do NOT use NVFP4 mirrors (Blackwell-only; L4 is Ada sm_89).
The `model_sources` field is intentionally left empty in `kernel-metadata.json` because the CLI requires a
full instance-version path that resolves only from the model owner's account — attach the model in the UI once.

**GPU note:** metadata keeps `enable_gpu: false` per branch policy. The runner auto-detects CUDA:
with GPUs it launches vLLM (TP=4) and runs the full pipeline; without GPUs it falls back to a
built-in template MockLLM so the notebook still produces a schema-valid submission and exercises
the whole harness (useful smoke mode). Rerun wall clock: 12h with cumulative atomic
`/kaggle/working/submission.json` writes after every task + a watchdog.

In [ ]:
import os, sys, json, time, shutil, subprocess

START_TS = time.time()
WORKING = "/kaggle/working"
H3 = os.path.join(WORKING, "h3")
os.makedirs(os.path.join(H3, "harness"), exist_ok=True)

# 12h rerun limit minus a 10-minute safety buffer
END_TS = START_TS + 12 * 3600 - 600

RERUN = os.getenv("KAGGLE_IS_COMPETITION_RERUN", "") != ""

DATA_DIR = None
for p in (
    "/kaggle/input/competitions/arc-prize-2026-arc-agi-2",
    "/kaggle/input/arc-prize-2026-arc-agi-2",
):
    if os.path.isdir(p):
        DATA_DIR = p
        break

MODEL_DIR = None
for p in (
    "/kaggle/input/qwen3-8-27b-fp8",
    "/kaggle/input/qwen3-8-27b-fp8-official",
    "/kaggle/input/qwen-qwen3-8-27b-fp8",
    "/kaggle/input/qwen3-8-27b-fp8-hf-snapshot",
    "/kaggle/input/qwen3-8-27b-bf16",
    "/kaggle/input/qwen3-8-27b",
):
    if os.path.isdir(p):
        MODEL_DIR = p
        break

def _has_gpu():
    try:
        if shutil.which("nvidia-smi") is None:
            return False
        return subprocess.call(["nvidia-smi"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL) == 0
    except Exception:
        return False

HAS_GPU = _has_gpu()
print("rerun mode:", RERUN)
print("data dir:", DATA_DIR)
print("model dir:", MODEL_DIR)
print("gpu available:", HAS_GPU)
print("wall budget (s):", END_TS - START_TS)


In [ ]:
%%writefile /kaggle/working/h3/harness/types.py
"""Core data types for the H3 code-synthesis + verified-set disambiguation harness."""
from __future__ import annotations

from dataclasses import dataclass, field
from typing import Any, Optional

import numpy as np

Grid = np.ndarray  # 2-D array of ints 0..9


@dataclass(frozen=True)
class Task:
    task_id: str
    train: list[tuple[Grid, Grid]]
    test: list[Grid]

    @property
    def n_pairs(self) -> int:
        return len(self.train)

    @property
    def n_test(self) -> int:
        return len(self.test)


@dataclass
class ExecResult:
    ok: bool
    output: Optional[Grid] = None
    error: Optional[str] = None
    timed_out: bool = False
    ms: float = 0.0


@dataclass
class PairScore:
    """Outcome of running one program on one (input, expected_output) pair."""

    exact: bool
    soft: float  # mean cell accuracy when shapes match, else 0.0
    shape_ok: bool
    error: Optional[str] = None  # sandbox/execution error, if any
    diff: Optional[str] = None  # pred/truth diagnostic grid text


@dataclass
class ProgramEval:
    per_pair: list[PairScore]
    verified: bool  # all pairs exact
    soft_mean: float
    n_exec_errors: int = 0


@dataclass
class Candidate:
    """One synthesized program plus everything we know about it."""

    code: str
    source: str  # provenance tag, e.g. "e0i3" (expert 0, iteration 3)
    eval: ProgramEval
    test_outputs: list[Optional[Grid]] = field(default_factory=list)
    adj_vote: int = 0  # +1 when an adjudication/LOO program corroborates this behavior
    loo_corroboration: float = 0.0  # fraction of LOO folds whose verified program matches this behavior

    @property
    def verified(self) -> bool:
        return self.eval.verified

    @property
    def soft_mean(self) -> float:
        return self.eval.soft_mean


@dataclass
class ClusterScore:
    rep_index: int
    test_output: Optional[Grid]
    score: float
    support: float  # independence-weighted member count
    members: list[int]
    features: dict[str, float]


@dataclass
class DisambiguationResult:
    attempt_1: Optional[Grid]
    attempt_2: Optional[Grid]
    clusters: list[ClusterScore]
    features: list[dict[str, Any]]  # per-candidate feature dicts (for caching/ablation)
    agreement: Optional[np.ndarray] = None  # pairwise probe-agreement matrix
    notes: list[str] = field(default_factory=list)


@dataclass
class LLMResponse:
    text: str
    prompt_tokens: int
    completion_tokens: int
    model: str = "mock"


@dataclass
class TaskResult:
    task_id: str
    attempts: list[tuple[Optional[Grid], Optional[Grid]]]  # per test input
    n_candidates: int
    n_verified: int
    n_distinct_verified_behaviors: int
    llm_calls: int
    prompt_tokens: int
    completion_tokens: int
    wall_s: float
    disambiguation: Optional[DisambiguationResult] = None
    extras: dict[str, Any] = field(default_factory=dict)


In [ ]:
%%writefile /kaggle/working/h3/harness/grid.py
"""Grid utilities: parsing/formatting, the D4 symmetry group, color permutations,
shape-law inference, data-consistency checks and probe perturbations.

Everything here is pure numpy/stdlib so the whole disambiguation layer is
testable offline (no GPU, no LLM).
"""
from __future__ import annotations

import itertools
from collections import Counter, deque
from fractions import Fraction
from typing import Iterable, Optional, Sequence

import numpy as np

from .types import Grid

# ---------------------------------------------------------------------------
# basic grid handling
# ---------------------------------------------------------------------------


def to_grid(x: Sequence[Sequence[int]] | np.ndarray) -> Grid:
    """Convert nested lists / arrays to a 2-D int64 numpy grid."""
    arr = np.asarray(x)
    if arr.ndim != 2:
        if arr.ndim == 1 and arr.size == 0:
            arr = arr.reshape(0, 0)
        else:
            raise ValueError(f"expected 2-D grid, got shape {arr.shape}")
    if arr.size and (arr.min() < 0 or arr.max() > 9):
        raise ValueError("grid values must be in 0..9")
    return arr.astype(np.int64, copy=False)


def grid_key(g: Grid) -> tuple:
    """Hashable canonical key for a grid."""
    a = np.asarray(g)
    return (a.shape[0], a.shape[1], a.tobytes())


def grids_equal(a: Optional[Grid], b: Optional[Grid]) -> bool:
    if a is None or b is None:
        return a is None and b is None
    a, b = np.asarray(a), np.asarray(b)
    return a.shape == b.shape and bool(np.array_equal(a, b))


def format_csv(g: Grid) -> str:
    """CSV rendering (benchmarked best of 9 formats; a bad format costs ~10%)."""
    return "\n".join(",".join(str(int(c)) for c in row) for row in np.asarray(g))


def parse_grid_lenient(text: str) -> Optional[Grid]:
    """Tolerantly parse a grid out of model output text.

    Accepts CSV rows, space-separated digits, bracketed JSON-ish rows. Returns
    the LAST plausible rectangular block (models tend to restate the final
    answer last), or None.
    """
    import json
    import re

    text = text.strip()
    # try whole-text JSON first
    for cand in (text, re.sub(r"^[^{\[]*", "", text)):
        try:
            obj = json.loads(cand)
            if isinstance(obj, list) and obj and isinstance(obj[0], list):
                return to_grid(obj)
        except Exception:
            pass

    # first balanced-looking nested JSON array anywhere in the text
    m = re.search(r"\[\s*\[.*?\]\s*\]", text, re.DOTALL)
    if m:
        try:
            obj = json.loads(m.group(0))
            if (
                isinstance(obj, list)
                and obj
                and all(isinstance(r, list) for r in obj)
                and len({len(r) for r in obj}) == 1
                and all(isinstance(v, int) and 0 <= v <= 9 for r in obj for v in r)
            ):
                return to_grid(obj)
        except Exception:
            pass

    rows: list[list[int]] = []
    blocks: list[list[list[int]]] = []
    for line in text.splitlines():
        s = line.strip().strip("`").strip()
        s = s.replace("[", " ").replace("]", " ").replace(",", " ")
        toks = s.split()
        if toks and all(re.fullmatch(r"\d", t) for t in toks):
            rows.append([int(t) for t in toks])
        else:
            if rows:
                blocks.append(rows)
                rows = []
    if rows:
        blocks.append(rows)

    # keep rectangular blocks; return the last one
    for blk in reversed(blocks):
        widths = {len(r) for r in blk}
        if len(widths) == 1 and blk:
            return to_grid(blk)
    return None


def background_color(g: Grid) -> int:
    vals, counts = np.unique(np.asarray(g), return_counts=True)
    return int(vals[int(np.argmax(counts))])


# ---------------------------------------------------------------------------
# D4 symmetry group
# ---------------------------------------------------------------------------
# An element is (k, f): apply horizontal-axis flip (up-down) first if f, then
# rotate 90 deg counter-clockwise k times. The 8 elements form the dihedral
# group D4 acting on rectangular grids.

D4_ELEMENTS: list[tuple[int, bool]] = [
    (0, False),  # identity
    (1, False),  # rot90 CCW
    (2, False),  # rot180
    (3, False),  # rot270 CCW (= rot90 CW)
    (0, True),   # flip up-down
    (1, True),   # flip then rot90 (= transpose)
    (2, True),   # flip then rot180 (= flip left-right)
    (3, True),   # flip then rot270 (= anti-transpose)
]

D4_NAMES = ["id", "rot90", "rot180", "rot270", "flipud", "transpose", "fliplr", "antitranspose"]


def d4_name(elem: tuple[int, bool]) -> str:
    return D4_NAMES[D4_ELEMENTS.index(elem)]


def apply_d4(g: Grid, elem: tuple[int, bool]) -> Grid:
    k, f = elem
    x = np.asarray(g)
    if f:
        x = x[::-1, :]
    if k:
        x = np.rot90(x, k)
    return np.ascontiguousarray(x)


def apply_d4_inv(g: Grid, elem: tuple[int, bool]) -> Grid:
    k, f = elem
    x = np.asarray(g)
    if k:
        x = np.rot90(x, -k)
    if f:
        x = x[::-1, :]
    return np.ascontiguousarray(x)


def compose_d4(a: tuple[int, bool], b: tuple[int, bool]) -> tuple[int, bool]:
    """Return c with apply(apply(x,b),a) == apply(x,c)."""
    ka, fa = a
    kb, fb = b
    # rot(k) o flip = flip o rot(-k);  (ka,fa)*(kb,fb)
    k = (ka + (-kb if fa else kb)) % 4
    return (k, fa ^ fb)


def invert_d4(elem: tuple[int, bool]) -> tuple[int, bool]:
    k, f = elem
    if not f:
        return ((-k) % 4, False)
    return (k, True)  # flip o rot(k) inverted = rot(-k) o flip = flip o rot(k)


# ---------------------------------------------------------------------------
# color permutations
# ---------------------------------------------------------------------------


def apply_color_perm(g: Grid, perm: dict[int, int]) -> Grid:
    x = np.asarray(g).copy()
    for a, b in perm.items():
        x[np.asarray(g) == a] = b
    return x


def color_profile(pairs: list[tuple[Grid, Grid]], color: int) -> tuple:
    """Usage signature of a color across train pairs: sorted (count_in, count_out)."""
    prof = []
    for xi, yi in pairs:
        prof.append((int(np.sum(xi == color)), int(np.sum(yi == color))))
    return tuple(sorted(prof))


def candidate_color_perms(
    pairs: list[tuple[Grid, Grid]], max_perms: int = 6
) -> list[dict[int, int]]:
    """Color transpositions that are plausible task symmetries.

    A transposition (c1 c2) is *candidate* when both colors have identical
    usage profiles across train pairs (they are interchangeable as far as the
    data can tell) and the permuted relation set stays consistent with the
    original one (checked by the caller via relations_consistent). Compositions
    of disjoint transpositions are added when cheap.
    """
    colors = set()
    for xi, yi in pairs:
        colors.update(np.unique(xi).tolist())
        colors.update(np.unique(yi).tolist())
    colors = sorted(int(c) for c in colors)
    profiles = {c: color_profile(pairs, c) for c in colors}

    groups: dict[tuple, list[int]] = {}
    for c, p in profiles.items():
        groups.setdefault(p, []).append(c)

    perms: list[dict[int, int]] = []
    for p, cs in groups.items():
        if len(cs) < 2:
            continue
        if all(sum(t) == 0 for t in p):
            # colors that never appear carry no information
            continue
        cs = sorted(cs)
        for a, b in itertools.combinations(cs, 2):
            perms.append({a: b, b: a})
            if len(perms) >= max_perms:
                return perms
        # 3-cycles for groups of >=3 interchangeable colors
        if len(cs) >= 3:
            a, b, c = cs[:3]
            perms.append({a: b, b: c, c: a})
            if len(perms) >= max_perms:
                return perms
    return perms[:max_perms]


# ---------------------------------------------------------------------------
# data-level consistency of pseudo-labeled (augmented) relations
# ---------------------------------------------------------------------------


def relations_consistent(relations: Iterable[tuple[Grid, Grid]]) -> bool:
    """False when the same input is related to two different outputs."""
    seen: dict[tuple, tuple] = {}
    for x, y in relations:
        kx = grid_key(x)
        ky = grid_key(y)
        if kx in seen and seen[kx] != ky:
            return False
        seen[kx] = ky
    return True


def plausible_d4(pairs: list[tuple[Grid, Grid]]) -> list[tuple[int, bool]]:
    """D4 elements whose pseudo-labels do not contradict the train data.

    Element s survives when {(s.x_i -> s.y_i)} unioned with the original pairs
    is a consistent relation. This is a *data-level* screen only; whether the
    true rule actually commutes with s is decided later by pool-majority
    resolvability (see disambiguate.py).
    """
    out = []
    for elem in D4_ELEMENTS:
        rel = list(pairs) + [(apply_d4(x, elem), apply_d4(y, elem)) for x, y in pairs]
        if relations_consistent(rel):
            out.append(elem)
    return out


def plausible_color_perms(
    pairs: list[tuple[Grid, Grid]], max_perms: int = 6
) -> list[dict[int, int]]:
    out = []
    for perm in candidate_color_perms(pairs, max_perms=max_perms * 2):
        rel = list(pairs) + [
            (apply_color_perm(x, perm), apply_color_perm(y, perm)) for x, y in pairs
        ]
        if relations_consistent(rel):
            out.append(perm)
        if len(out) >= max_perms:
            break
    return out


# ---------------------------------------------------------------------------
# shape laws
# ---------------------------------------------------------------------------


class ShapeLaw:
    """Predicts output shape from input shape. None => no prediction."""

    def __init__(self, name: str, fn):
        self.name = name
        self._fn = fn

    def predict(self, h: int, w: int) -> Optional[tuple[int, int]]:
        return self._fn(h, w)

    def __repr__(self) -> str:  # pragma: no cover
        return f"ShapeLaw({self.name})"


def _affine_laws(pairs: list[tuple[Grid, Grid]]) -> list[ShapeLaw]:
    """Enumerate small exact affine laws out_h = a*h + b*w + c (and for w)."""
    cand_vals = [Fraction(v) for v in (0, 0.5, 1, 1.5, 2, 3, -0.5, -1, -2)]
    laws_h: list[tuple[Fraction, Fraction, Fraction]] = []
    laws_w: list[tuple[Fraction, Fraction, Fraction]] = []
    shapes = [(x.shape[0], x.shape[1], y.shape[0], y.shape[1]) for x, y in pairs]
    for a in cand_vals:
        for b in cand_vals:
            for c in (Fraction(v) for v in range(-6, 7)):
                if all(a * h + b * w + c == oh for h, w, oh, _ in shapes):
                    laws_h.append((a, b, c))
                if all(a * h + b * w + c == ow for h, w, _, ow in shapes):
                    laws_w.append((a, b, c))
    laws = []
    for (ah, bh, ch) in laws_h:
        for (aw, bw, cw) in laws_w:
            def fn(h, w, _h=(ah, bh, ch), _w=(aw, bw, cw)):
                oh = _h[0] * h + _h[1] * w + _h[2]
                ow = _w[0] * h + _w[1] * w + _w[2]
                if oh <= 0 or ow <= 0 or oh.denominator != 1 or ow.denominator != 1:
                    return None
                if oh > 64 or ow > 64:
                    return None
                return (int(oh), int(ow))

            laws.append(ShapeLaw(f"affine(h:{ah},{bh},{ch};w:{aw},{bw},{cw})", fn))
    return laws


def consistent_shape_laws(pairs: list[tuple[Grid, Grid]]) -> list[ShapeLaw]:
    """All shape laws from a small interpretable family that fit every train pair.

    Family: identity, constant, rational per-dimension scale, affine. The
    violation test only fires when *every* consistent law disagrees with a
    program's output shape, which keeps the signal sound on small data.
    """
    if not pairs:
        return []
    laws: list[ShapeLaw] = []
    in_shapes = {(x.shape[0], x.shape[1]) for x, _ in pairs}
    out_shapes = {(y.shape[0], y.shape[1]) for _, y in pairs}

    if all(x.shape == y.shape for x, y in pairs):
        laws.append(ShapeLaw("identity", lambda h, w: (h, w)))
    if len(out_shapes) == 1:
        (oh, ow), = out_shapes
        laws.append(ShapeLaw(f"const({oh},{ow})", lambda h, w, _o=(oh, ow): _o))
    scales = []
    for qh in (Fraction(1), Fraction(1, 2), Fraction(2), Fraction(1, 3), Fraction(3),
                Fraction(2, 3), Fraction(3, 2), Fraction(1, 4), Fraction(4)):
        for qw in (Fraction(1), Fraction(1, 2), Fraction(2), Fraction(1, 3), Fraction(3),
                   Fraction(2, 3), Fraction(3, 2), Fraction(1, 4), Fraction(4)):
            if qh == 1 and qw == 1:
                continue  # covered by SameLaw
            ok = True
            for x, y in pairs:
                h, w = x.shape
                if h * qh != y.shape[0] or w * qw != y.shape[1]:
                    ok = False
                    break
            if ok:
                scales.append((qh, qw))
    if scales and len(in_shapes) >= 2:
        for qh, qw in scales[:4]:
            laws.append(
                ShapeLaw(
                    f"scale({qh},{qw})",
                    lambda h, w, _q=(qh, qw): (
                        (int(h * _q[0]), int(w * _q[1]))
                        if (h * _q[0]).denominator == 1 and (w * _q[1]).denominator == 1
                        and 0 < h * _q[0] <= 64 and 0 < w * _q[1] <= 64
                        else None
                    ),
                )
            )
    if len({(x.shape[0], x.shape[1]) for x, _ in pairs}) >= 3:
        laws.extend(_affine_laws(pairs))
    return laws


def shape_law_violation(
    laws: Sequence[ShapeLaw], in_shape: tuple[int, int], out_shape: Optional[tuple[int, int]]
) -> bool:
    """True when out_shape contradicts EVERY consistent law (unanimous vote)."""
    if out_shape is None or not laws:
        return False
    preds = []
    for law in laws:
        p = law.predict(*in_shape)
        if p is not None:
            preds.append(tuple(p))
    if not preds:
        return False
    return all(p != tuple(out_shape) for p in preds)


# ---------------------------------------------------------------------------
# probes / perturbations (behavioral clustering inputs)
# ---------------------------------------------------------------------------


def pad_border(g: Grid, color: Optional[int] = None, width: int = 1) -> Grid:
    x = np.asarray(g)
    c = background_color(x) if color is None else color
    return np.pad(x, width, mode="constant", constant_values=c).astype(x.dtype, copy=False)


def crop_border(g: Grid, width: int = 1) -> Optional[Grid]:
    x = np.asarray(g)
    if x.shape[0] <= 2 * width or x.shape[1] <= 2 * width:
        return None
    return x[width:-width, width:-width]


def perturb_cells(g: Grid, rng: np.random.Generator, n_cells: int = 1) -> Grid:
    """Recolor n random cells to a different random color."""
    x = np.asarray(g).copy()
    h, w = x.shape
    for _ in range(n_cells):
        r, c = int(rng.integers(0, h)), int(rng.integers(0, w))
        choices = [v for v in range(10) if v != x[r, c]]
        x[r, c] = int(rng.choice(choices))
    return x


# ---------------------------------------------------------------------------
# tiny structural analysis used in prompts (Gre24-style extra representation)
# ---------------------------------------------------------------------------


def components(g: Grid, include_background: bool = False) -> list[dict]:
    """4-connected components as dicts {color, size, bbox}."""
    x = np.asarray(g)
    h, w = x.shape
    seen = np.zeros_like(x, dtype=bool)
    bg = background_color(x)
    out = []
    for i in range(h):
        for j in range(w):
            if seen[i, j] or (x[i, j] == bg and not include_background):
                continue
            color = int(x[i, j])
            q = deque([(i, j)])
            seen[i, j] = True
            cells = []
            while q:
                r, c = q.popleft()
                cells.append((r, c))
                for dr, dc in ((1, 0), (-1, 0), (0, 1), (0, -1)):
                    nr, nc = r + dr, c + dc
                    if 0 <= nr < h and 0 <= nc < w and not seen[nr, nc] and x[nr, nc] == color:
                        seen[nr, nc] = True
                        q.append((nr, nc))
            rs = [r for r, _ in cells]
            cs = [c for _, c in cells]
            out.append(
                {
                    "color": color,
                    "size": len(cells),
                    "bbox": (min(rs), min(cs), max(rs), max(cs)),
                }
            )
    return out


def analysis_block(pairs: list[tuple[Grid, Grid]], test_inputs: list[Grid]) -> str:
    """Compact per-grid structural summary appended to synthesis prompts."""
    lines = []
    for idx, (x, y) in enumerate(pairs):
        for tag, g in (("in", x), ("out", y)):
            comps = components(g)
            palette = sorted(set(int(v) for v in np.unique(g)))
            lines.append(
                f"pair{idx + 1}.{tag}: {g.shape[0]}x{g.shape[1]} bg={background_color(g)} "
                f"palette={palette} objects={len(comps)} "
                f"obj_sizes={sorted((c['size'] for c in comps), reverse=True)[:6]}"
            )
    for idx, x in enumerate(test_inputs):
        comps = components(x)
        palette = sorted(set(int(v) for v in np.unique(x)))
        lines.append(
            f"test{idx + 1}.in: {x.shape[0]}x{x.shape[1]} bg={background_color(x)} "
            f"palette={palette} objects={len(comps)} "
            f"obj_sizes={sorted((c['size'] for c in comps), reverse=True)[:6]}"
        )
    return "\n".join(lines)


In [ ]:
%%writefile /kaggle/working/h3/harness/extraction.py
"""Tolerant extraction of a `transform` program from model output.

The model is asked for a single markdown-fenced python block, but thinking-mode
models routinely emit extra prose, several blocks, or forget fences entirely
(vLLM structured output is unreliable in thinking mode, vLLM#35700), so we
parse tolerantly and prefer the LAST block that defines `transform`.
"""
from __future__ import annotations

import re
from typing import Optional

_FENCE_RE = re.compile(r"^[\s>]*(```+|~~~+)[ \t]*([A-Za-z0-9_+-]*)[ \t]*\r?$")


def extract_code_blocks(text: str) -> list[str]:
    """Return all fenced code block bodies, in order. Handles unclosed fences."""
    blocks: list[str] = []
    lines = text.splitlines()
    i = 0
    while i < len(lines):
        m = _FENCE_RE.match(lines[i])
        if m:
            fence = m.group(1)[0] * 3  # normalize ```` -> ```
            body: list[str] = []
            i += 1
            closed = False
            while i < len(lines):
                m2 = _FENCE_RE.match(lines[i])
                if m2 and m2.group(1)[0] * 3 == fence:
                    closed = True
                    i += 1
                    break
                body.append(lines[i])
                i += 1
            blocks.append("\n".join(body))
            if not closed:
                break
        else:
            i += 1
    return blocks


def pick_program(text: str) -> Optional[str]:
    """Pick the best candidate program source from a model response.

    Preference order:
    1. last fenced block containing `def transform`
    2. last fenced block containing any `def `
    3. raw text, when it itself contains `def transform`
    """
    if not text:
        return None
    blocks = extract_code_blocks(text)
    with_transform = [b for b in blocks if re.search(r"^\s*def\s+transform\b", b, re.M)]
    if with_transform:
        return with_transform[-1].strip() + "\n"
    with_def = [b for b in blocks if re.search(r"^\s*def\s+\w+", b, re.M)]
    if with_def:
        return with_def[-1].strip() + "\n"
    if re.search(r"^\s*def\s+transform\b", text, re.M):
        return text.strip() + "\n"
    return None


def compiles(code: str) -> bool:
    try:
        compile(code, "<candidate>", "exec")
        return True
    except SyntaxError:
        return False


In [ ]:
%%writefile /kaggle/working/h3/harness/_sandbox_worker.py
"""Sandbox worker process (runs under `python -I` with a clean environment).

Protocol: reads newline-delimited JSON jobs on stdin:
    {"id": int, "code": str, "grid": [[int]]}
Writes one JSON result line per job to stdout (flushed immediately):
    {"id": int, "ok": bool, "out": [[int]]|null, "err": str|null, "ms": float}

The parent enforces a per-job deadline by killing this process when the next
result line does not arrive in time, then resumes the remaining jobs in a
fresh worker (see sandbox.py). On POSIX we additionally set SIGALRM per job
and RLIMIT_AS as defense in depth.
"""
from __future__ import annotations

import io
import json
import sys
import time

MAX_OUTPUT_CELLS = 64 * 64
MAX_CODE_BYTES = 128 * 1024
MAX_ERR_CHARS = 600


def _setup_posix_limits() -> None:
    try:
        import resource

        resource.setrlimit(resource.RLIMIT_AS, (2 << 30, 2 << 30))  # 2 GB
        resource.setrlimit(resource.RLIMIT_NPROC, (256, 256))
    except Exception:
        pass


def _setup_posix_alarm(timeout_s: float):
    import signal

    class _JobTimeout(BaseException):
        pass

    def handler(signum, frame):
        raise _JobTimeout()

    signal.signal(signal.SIGALRM, handler)
    signal.setitimer(signal.ITIMER_REAL, timeout_s)
    return _JobTimeout


def _cancel_posix_alarm() -> None:
    import signal

    signal.setitimer(signal.ITIMER_REAL, 0)


def main() -> None:
    is_posix = sys.platform != "win32"
    if is_posix:
        _setup_posix_limits()

    import numpy as np

    try:
        np.seterr(all="ignore")
    except Exception:
        pass

    out_write = sys.stdout
    code_cache: dict[int, dict] = {}
    job_timeout_s = 5.0

    for line in sys.stdin:
        line = line.strip()
        if not line:
            continue
        try:
            job = json.loads(line)
        except Exception:
            continue
        if job.get("op") == "config":
            job_timeout_s = float(job.get("timeout_s", job_timeout_s))
            continue
        jid = job.get("id")
        t0 = time.perf_counter()
        result = {"id": jid, "ok": False, "out": None, "err": None, "ms": 0.0}
        alarm_exc = None
        try:
            code = job["code"]
            if len(code.encode("utf-8", "replace")) > MAX_CODE_BYTES:
                raise ValueError("program too large")
            grid = np.array(job["grid"], dtype=np.int64)
            if grid.ndim != 2:
                raise ValueError("input grid must be 2-D")

            h = hash(code)
            ns = code_cache.get(h)
            if ns is None:
                ns = {"__builtins__": __builtins__}
                buf = io.StringIO()
                old = sys.stdout
                sys.stdout = buf
                try:
                    exec(compile(code, "<candidate>", "exec"), ns)
                finally:
                    sys.stdout = old
                if not callable(ns.get("transform")):
                    raise ValueError("program does not define callable transform(grid)")
                code_cache[h] = ns

            if is_posix:
                alarm_exc = _setup_posix_alarm(job_timeout_s)
            try:
                old = sys.stdout
                sys.stdout = io.StringIO()
                try:
                    out = ns["transform"](grid.copy())
                finally:
                    sys.stdout = old
            finally:
                if is_posix:
                    _cancel_posix_alarm()

            out = np.asarray(out)
            if out.dtype == bool:
                out = out.astype(np.int64)
            if np.issubdtype(out.dtype, np.floating):
                if out.size and not np.all(np.isfinite(out)):
                    raise ValueError("output contains non-finite values")
                out = np.rint(out).astype(np.int64)
            out = out.astype(np.int64, copy=False)
            if out.ndim != 2:
                raise ValueError(f"output must be 2-D, got ndim={out.ndim}")
            if out.size > MAX_OUTPUT_CELLS:
                raise ValueError(f"output too large: {out.shape}")
            if out.size and (out.min() < -999 or out.max() > 999):
                raise ValueError("output values out of sane range")
            result["ok"] = True
            result["out"] = out.tolist()
        except BaseException as e:  # noqa: BLE001 - sandbox must never crash
            if alarm_exc is not None and isinstance(e, alarm_exc):
                result["err"] = "timeout"
            else:
                result["err"] = f"{type(e).__name__}: {e}"[:MAX_ERR_CHARS]
        result["ms"] = round((time.perf_counter() - t0) * 1000, 2)
        out_write.write(json.dumps(result) + "\n")
        out_write.flush()


if __name__ == "__main__":
    main()


In [ ]:
%%writefile /kaggle/working/h3/harness/sandbox.py
"""Subprocess-isolated sandbox for model-generated programs.

Safety model:
- worker runs under `python -I` (isolated: ignores env vars & user site) with a
  minimal clean environment (no CUDA, no user env);
- parent-side per-job deadline: the reader kills and restarts the worker when
  the next result line does not arrive in time, so infinite loops cannot wedge
  a batch (portable: works on Windows dev machines and Linux/Kaggle);
- POSIX-only defense in depth: SIGALRM per job + RLIMIT_AS 2GB inside worker;
- output caps: grids > 64x64 rejected, error strings truncated, code size capped.

The worker reads NDJSON jobs from stdin and streams NDJSON results to stdout,
so a whole verification/disambiguation battery (hundreds of grid evaluations)
runs in ONE subprocess, amortizing interpreter+numpy startup (~0.2-0.4 s).
"""
from __future__ import annotations

import json
import os
import queue
import subprocess
import sys
import tempfile
import threading
import time
from collections import Counter
from pathlib import Path
from typing import Callable, Optional, Sequence

import numpy as np

from .types import ExecResult, Grid

_WORKER = Path(__file__).with_name("_sandbox_worker.py")


def _clean_env(tmpdir: str) -> dict[str, str]:
    env = {
        "PYTHONHASHSEED": "0",
        "PYTHONIOENCODING": "utf-8",
        "PYTHONDONTWRITEBYTECODE": "1",
        "CUDA_VISIBLE_DEVICES": "",  # never let sandboxed code touch the GPU
        "TMPDIR": tmpdir,
        "TEMP": tmpdir,
        "TMP": tmpdir,
    }
    if os.name == "nt":
        for k in ("SYSTEMROOT", "COMSPEC", "PATHEXT"):
            if k in os.environ:
                env[k] = os.environ[k]
        env["HOME"] = tmpdir
        env["USERPROFILE"] = tmpdir
    else:
        env["HOME"] = tmpdir
        # keep dynamic loader paths so cv2/scipy work when installed (Kaggle)
        for k in ("LD_LIBRARY_PATH", "PATH"):
            if k in os.environ:
                env[k] = os.environ[k]
    return env


class Sandbox:
    """Batch executor: `run_batch([(code, grid), ...]) -> [ExecResult, ...]`."""

    def __init__(
        self,
        per_job_timeout: float = 2.0,
        startup_timeout: float = 30.0,
        python_exe: Optional[str] = None,
    ):
        self.per_job_timeout = per_job_timeout
        self.startup_timeout = startup_timeout
        self.python_exe = python_exe or sys.executable
        self._tmp: Optional[tempfile.TemporaryDirectory] = None
        self.worker_restarts = 0

    # -- lifecycle ----------------------------------------------------------

    def _tmpdir(self) -> str:
        if self._tmp is None:
            self._tmp = tempfile.TemporaryDirectory(prefix="h3_sandbox_")
        return self._tmp.name

    def close(self) -> None:
        if self._tmp is not None:
            try:
                self._tmp.cleanup()
            except OSError:
                pass
            self._tmp = None

    def __enter__(self) -> "Sandbox":
        return self

    def __exit__(self, *exc) -> None:
        self.close()

    # -- public API ---------------------------------------------------------

    def run(self, code: str, grid: Grid) -> ExecResult:
        return self.run_batch([(code, grid)])[0]

    def run_batch(
        self,
        jobs: Sequence[tuple[str, Grid]],
        on_result: Optional[Callable[[int, ExecResult], None]] = None,
    ) -> list[ExecResult]:
        n = len(jobs)
        results: list[Optional[ExecResult]] = [None] * n
        pending: list[int] = list(range(n))
        poison: Counter[int] = Counter()
        restarts = 0
        max_restarts = 2 * n + 4
        tmpdir = self._tmpdir()

        while pending and restarts <= max_restarts:
            wedge = self._run_worker(jobs, pending, results, tmpdir, on_result)
            restarts += 1
            self.worker_restarts += 1
            if wedge is not None:
                idx, kind = wedge
                if kind == "crash":
                    poison[idx] += 1
                    if poison[idx] >= 2:
                        results[idx] = ExecResult(
                            ok=False, output=None, error="worker died on job twice", timed_out=False
                        )
                    # poison==1: leave result None -> job retried in a fresh worker
                # kind == "timeout": result already recorded by _run_worker
            pending = [i for i in pending if results[i] is None]

        for i in pending:  # restart budget exceeded
            results[i] = ExecResult(ok=False, error="sandbox restart budget exceeded", timed_out=True)
        out = [r if r is not None else ExecResult(ok=False, error="missing result") for r in results]
        return out

    # -- internals ----------------------------------------------------------

    def _run_worker(
        self,
        jobs: Sequence[tuple[str, Grid]],
        pending: list[int],
        results: list[Optional[ExecResult]],
        tmpdir: str,
        on_result: Optional[Callable[[int, ExecResult], None]],
    ) -> Optional[tuple[int, str]]:
        """Run one worker over `pending` jobs.

        Returns `(job_index, kind)` for the job that wedged the worker, where
        kind is "timeout" (parent deadline fired; result already recorded) or
        "crash" (worker died before producing a result), or None on clean run.
        """
        proc = subprocess.Popen(
            [self.python_exe, "-I", "-u", str(_WORKER)],
            stdin=subprocess.PIPE,
            stdout=subprocess.PIPE,
            stderr=subprocess.DEVNULL,
            cwd=tmpdir,
            env=_clean_env(tmpdir),
        )
        lines_q: queue.Queue = queue.Queue()

        def reader() -> None:
            assert proc.stdout is not None
            try:
                for raw in proc.stdout:
                    lines_q.put(raw)
            except Exception:
                pass
            finally:
                lines_q.put(None)  # EOF sentinel

        def writer() -> None:
            assert proc.stdin is not None
            try:
                proc.stdin.write(
                    (json.dumps({"op": "config", "timeout_s": self.per_job_timeout}) + "\n").encode()
                )
                for idx in pending:
                    code, grid = jobs[idx]
                    job = {"id": idx, "code": code, "grid": np.asarray(grid).tolist()}
                    proc.stdin.write((json.dumps(job) + "\n").encode())
                proc.stdin.flush()
                proc.stdin.close()
            except (BrokenPipeError, OSError):
                pass

        rt = threading.Thread(target=reader, daemon=True)
        wt = threading.Thread(target=writer, daemon=True)
        rt.start()
        wt.start()

        expected = list(pending)
        pos = 0
        first_line = True
        wedge: Optional[tuple[int, str]] = None
        try:
            while pos < len(expected):
                deadline = self.per_job_timeout + (self.startup_timeout if first_line else 0.5)
                try:
                    raw = lines_q.get(timeout=deadline)
                except queue.Empty:
                    # worker wedged on job expected[pos] -> kill; caller reruns rest
                    wedge = (expected[pos], "timeout")
                    results[wedge[0]] = ExecResult(ok=False, error="timeout", timed_out=True)
                    if on_result:
                        on_result(wedge[0], results[wedge[0]])
                    break
                first_line = False
                if raw is None:  # EOF: worker died before emitting a result
                    wedge = (expected[pos], "crash")
                    break
                try:
                    res = json.loads(raw.decode("utf-8", "replace"))
                except Exception:
                    continue
                idx = int(res["id"])
                out = None
                if res.get("ok") and res.get("out") is not None:
                    out = np.array(res["out"], dtype=np.int64)
                er = ExecResult(
                    ok=bool(res.get("ok")),
                    output=out,
                    error=res.get("err"),
                    timed_out=(res.get("err") == "timeout"),
                    ms=float(res.get("ms", 0.0)),
                )
                if 0 <= idx < len(results):
                    results[idx] = er
                    if on_result:
                        on_result(idx, er)
                if pos < len(expected) and idx == expected[pos]:
                    pos += 1
        finally:
            if proc.poll() is None:
                try:
                    proc.kill()
                except OSError:
                    pass
            try:
                proc.wait(timeout=5)
            except Exception:
                pass
            wt.join(timeout=1)
        return wedge


class LocalExecutor:
    """In-process executor for tests only (NO isolation). Same API as Sandbox."""

    def __init__(self, per_job_timeout: float = 2.0):
        self.per_job_timeout = per_job_timeout
        self.worker_restarts = 0
        self._cache: dict[int, dict] = {}

    def run(self, code: str, grid: Grid) -> ExecResult:
        return self.run_batch([(code, grid)])[0]

    def run_batch(
        self,
        jobs: Sequence[tuple[str, Grid]],
        on_result: Optional[Callable[[int, ExecResult], None]] = None,
    ) -> list[ExecResult]:
        out: list[ExecResult] = []
        for i, (code, grid) in enumerate(jobs):
            t0 = time.perf_counter()
            try:
                h = hash(code)
                ns = self._cache.get(h)
                if ns is None:
                    ns = {}
                    import numpy as _np  # noqa: F401

                    exec(compile(code, "<candidate>", "exec"), ns)
                    if not callable(ns.get("transform")):
                        raise ValueError("no transform()")
                    self._cache[h] = ns
                res = ns["transform"](np.asarray(grid).copy())
                arr = np.asarray(res)
                if arr.ndim != 2:
                    raise ValueError("output not 2-D")
                arr = arr.astype(np.int64)
                if arr.size > 64 * 64:
                    raise ValueError("output too large")
                er = ExecResult(True, arr, None, False, (time.perf_counter() - t0) * 1000)
            except Exception as e:  # noqa: BLE001
                er = ExecResult(False, None, f"{type(e).__name__}: {e}", False, (time.perf_counter() - t0) * 1000)
            out.append(er)
            if on_result:
                on_result(i, er)
        return out

    def close(self) -> None:
        pass

    def __enter__(self) -> "LocalExecutor":
        return self

    def __exit__(self, *exc) -> None:
        self.close()


def os_name_check() -> str:  # pragma: no cover - trivial helper used in reports
    return os.name


In [ ]:
%%writefile /kaggle/working/h3/harness/oracle.py
"""Train-pair oracle: exact verification, soft scoring and diff diagnostics.

The train pairs are a free, exact, deterministic oracle for synthesized
programs. Land's held-out measurement (precision ~97%, recall ~43%) means
verification is a STRONG POSITIVE and a WEAK NEGATIVE signal, so nothing is
ever discarded for failing it: soft scores keep partial programs ranked
(see pool.py / disambiguate.py fallback path).
"""
from __future__ import annotations

from typing import Optional, Sequence

import numpy as np

from .sandbox import Sandbox
from .types import ExecResult, Grid, PairScore, ProgramEval


def score_output(pred: Optional[Grid], truth: Grid) -> PairScore:
    """Score one predicted grid against ground truth."""
    if pred is None:
        return PairScore(exact=False, soft=0.0, shape_ok=False, error="no output", diff=None)
    pred = np.asarray(pred)
    truth = np.asarray(truth)
    shape_ok = pred.shape == truth.shape
    if not shape_ok:
        return PairScore(
            exact=False,
            soft=0.0,
            shape_ok=False,
            error=f"shape mismatch: got {pred.shape}, expected {truth.shape}",
            diff=None,
        )
    eq = pred == truth
    exact = bool(eq.all())
    soft = float(np.nan_to_num(np.mean(eq)))
    return PairScore(exact=exact, soft=soft, shape_ok=True, error=None, diff=array_diff(pred, truth))


def array_diff(pred: Grid, truth: Grid) -> str:
    """Poetiq-style diagnostic: matching cells as-is, mismatches as 'pred/truth'."""
    pred = np.asarray(pred)
    truth = np.asarray(truth)
    lines = []
    for i in range(truth.shape[0]):
        row = []
        for j in range(truth.shape[1]):
            if pred[i, j] == truth[i, j]:
                row.append(str(int(truth[i, j])))
            else:
                row.append(f"{int(pred[i, j])}/{int(truth[i, j])}")
        lines.append(" ".join(row))
    return "\n".join(lines)


def evaluate_program(
    sandbox: Sandbox,
    code: str,
    pairs: Sequence[tuple[Grid, Grid]],
) -> ProgramEval:
    """Run a program over (input, output) pairs; exact verification + soft scores."""
    jobs = [(code, x) for x, _ in pairs]
    results: list[ExecResult] = sandbox.run_batch(jobs)
    per_pair: list[PairScore] = []
    n_err = 0
    for res, (_, truth) in zip(results, pairs):
        if not res.ok:
            n_err += 1
            per_pair.append(
                PairScore(exact=False, soft=0.0, shape_ok=False, error=res.error or "exec failed", diff=None)
            )
        else:
            per_pair.append(score_output(res.output, truth))
    verified = bool(per_pair) and all(ps.exact for ps in per_pair)
    soft_mean = float(np.mean([ps.soft for ps in per_pair])) if per_pair else 0.0
    return ProgramEval(per_pair=per_pair, verified=verified, soft_mean=soft_mean, n_exec_errors=n_err)


def run_on_inputs(
    sandbox: Sandbox, code: str, inputs: Sequence[Grid]
) -> list[Optional[Grid]]:
    """Run a program on raw inputs (no truth); failed runs yield None."""
    results = sandbox.run_batch([(code, x) for x in inputs])
    return [r.output if r.ok else None for r in results]


def build_feedback(eval_result: ProgramEval, pairs: Sequence[tuple[Grid, Grid]]) -> tuple[str, float]:
    """Textual feedback for the refinement loop + overall soft score."""
    parts: list[str] = []
    for i, ps in enumerate(eval_result.per_pair):
        if ps.exact:
            parts.append(f"Example #{i + 1}: CORRECT (exact match).")
            continue
        head = f"Example #{i + 1}: WRONG."
        if ps.error:
            parts.append(f"{head} Error: {ps.error}")
            continue
        block = [head]
        if not ps.shape_ok:
            block.append(ps.error or "shape mismatch")
        else:
            block.append(
                "Cells shown as `predicted/expected` where wrong, plain where right:"
            )
            block.append("```\n" + (ps.diff or "") + "\n```")
            block.append(f"Cell accuracy: {ps.soft:.2f}")
        parts.append("\n".join(block))
    return "\n\n".join(parts), eval_result.soft_mean


In [ ]:
%%writefile /kaggle/working/h3/harness/pool.py
"""Program pool: dedupe, behavior grouping, provenance tracking."""
from __future__ import annotations

import hashlib
import re
from typing import Optional, Sequence

import numpy as np

from .grid import grid_key
from .types import Candidate, Grid, ProgramEval


def source_hash(code: str) -> str:
    """Normalized-source identity: whitespace/comment-insensitive."""
    norm = re.sub(r"\s+", " ", re.sub(r"#.*", "", code)).strip()
    return hashlib.sha1(norm.encode()).hexdigest()[:16]


def behavior_key(eval_result: ProgramEval, test_outputs: Sequence[Optional[Grid]] = ()) -> tuple:
    """Two programs with the same behavior key are indistinguishable on all
    data we have (train outcomes incl. failures + test outputs)."""
    parts = []
    for ps in eval_result.per_pair:
        if ps.exact:
            parts.append(("exact",))
        elif ps.shape_ok:
            parts.append(("soft", round(ps.soft, 6)))
        else:
            parts.append(("err", (ps.error or "")[:40]))
    test_part = tuple(grid_key(o) if o is not None else None for o in test_outputs)
    return (tuple(parts), test_part)


class ProgramPool:
    """Keeps every behaviorally distinct program; groups by observable behavior."""

    def __init__(self) -> None:
        self.candidates: list[Candidate] = []
        self._by_source: dict[str, int] = {}

    def add(
        self,
        code: str,
        source: str,
        eval_result: ProgramEval,
        test_outputs: Optional[list[Optional[Grid]]] = None,
    ) -> tuple[int, bool]:
        """Returns (index, is_new). Source-identical duplicates are dropped;
        behavior duplicates are KEPT (independent corroborations matter for
        cluster support)."""
        sh = source_hash(code)
        if sh in self._by_source:
            return self._by_source[sh], False
        cand = Candidate(
            code=code,
            source=source,
            eval=eval_result,
            test_outputs=list(test_outputs) if test_outputs is not None else [],
        )
        idx = len(self.candidates)
        self.candidates.append(cand)
        self._by_source[sh] = idx
        return idx, True

    # -- views --------------------------------------------------------------

    def verified(self) -> list[int]:
        return [i for i, c in enumerate(self.candidates) if c.verified]

    def unverified(self) -> list[int]:
        return [i for i, c in enumerate(self.candidates) if not c.verified]

    def verified_behaviors(self) -> dict[tuple, list[int]]:
        """Group verified candidates by their test-output behavior.

        Candidates missing any test output form the None-keyed group and are
        excluded from attempt selection (but not from the pool).
        """
        groups: dict[tuple, list[int]] = {}
        for i in self.verified():
            c = self.candidates[i]
            if any(o is None for o in c.test_outputs):
                key = ("incomplete",)
            else:
                key = tuple(grid_key(o) for o in c.test_outputs)
            groups.setdefault(key, []).append(i)
        return groups

    def best_partial(self, n: int = 5) -> list[int]:
        """Top-n unverified candidates by soft score (never discard failures)."""
        unv = list(self.unverified())
        unv.sort(key=lambda i: (-self.candidates[i].soft_mean, len(self.candidates[i].code)))
        return unv[:n]

    def __len__(self) -> int:
        return len(self.candidates)


In [ ]:
%%writefile /kaggle/working/h3/harness/disambiguate.py
"""The disambiguation layer — the intellectual core of branch H3.

Given a pool of programs that all passed (or partially passed) the train-pair
oracle but DISAGREE on the test input, decide what to submit as attempt_1 and
attempt_2 — without any further LLM calls (everything here is numpy + sandbox
re-execution, i.e. effectively free in the token budget).

Signals (each justified in HYPOTHESIS.md):

* S1 equivariance     — a verified program should commute with the symmetries
                        the task plausibly exhibits (D4 + data-plausible color
                        permutations). Free pseudo-labeled held-out data:
                        (s·x_i -> s·y_i). Two-stage gating keeps this sound:
                        (a) data-level plausibility (pseudo-pairs must not
                        contradict train pairs), (b) pool-majority resolvability
                        (a symmetry only counts when a support-weighted majority
                        of distinct verified behaviors respects it — otherwise
                        the true rule itself breaks it and everyone would be
                        penalized equally, e.g. trivially-equivariant identity
                        programs must not outrank the true rule).
* S1c shape law       — affine/scale/const/identity shape laws fitted on train
                        pairs; a program whose test output shape contradicts
                        EVERY consistent law is penalized.
* S2 LOO corroboration— programs whose test behavior is reproduced by
                        held-out-fold resyntheses that passed their held-out
                        pair (LLM-in-the-loop, injected as `loo_map`; the
                        scoring logic itself is pure).
* S3 MDL prior        — shorter AST (comments/docstrings stripped) = higher
                        prior (CompressARC insight).
* S3b memorization    — literal train grids (or >=2 full rows) embedded in the
                        source = near-definitive overfit; heavy penalty.
* S4 behavioral
   clustering         — pairwise output agreement over a probe battery
                        (test inputs, their D4/color-perm variants, single-cell
                        mutations, border padding); cluster support = number of
                        independent programs agreeing.
* S5 soft fallback    — verification failure never discards: unverified
                        programs stay ranked by soft score (Land: precision
                        ~97%, recall ~43% => strong positive, weak negative).
* S6 determinism      — nondeterministic programs (hidden RNG) are penalized.
* S7 adjudication vote— an LLM adjudicator program (gated, budgeted) that
                        passes verification adds a vote to the matching
                        behavior (injected as `adj_votes`).

pass@2-aware selection: attempt_2 must be a DIFFERENT grid than attempt_1
(duplicating attempt_1 is strictly dominated under exact-match scoring), with a
shape-diversity bonus so the two attempts hedge different shape hypotheses.
"""
from __future__ import annotations

import ast
import re
from dataclasses import dataclass, field
from typing import Optional, Sequence

import numpy as np

from .grid import (
    D4_ELEMENTS,
    apply_color_perm,
    apply_d4,
    consistent_shape_laws,
    d4_name,
    grid_key,
    grids_equal,
    pad_border,
    perturb_cells,
    plausible_color_perms,
    plausible_d4,
    shape_law_violation,
)
from .types import Candidate, ClusterScore, DisambiguationResult, Grid, Task

# ---------------------------------------------------------------------------
# configuration
# ---------------------------------------------------------------------------

DEFAULT_WEIGHTS: dict[str, float] = {
    "verified": 2.0,      # gate-like bonus: verification is a strong positive
    "soft": 0.5,          # mean train cell accuracy (dominant when unverified)
    "support": 1.0,       # cluster support (behavioral consensus)
    "equiv": 0.8,         # symmetry equivariance (train pseudo-labels + test consistency)
    "shape": 1.5,         # penalty: contradicts every consistent shape law
    "memorization": 1.5,  # penalty: literal train grids in source
    "nondet": 1.0,        # penalty: nondeterministic outputs
    "mdl": 0.25,          # MDL prior (shorter = better)
    "robust": 0.2,        # fraction of probe battery that executes cleanly
    "loo": 0.6,           # LOO corroboration (when available)
    "adj": 0.6,           # adjudication vote (when available)
}

DEFAULT_ENABLE: dict[str, bool] = {
    "equiv": True,
    "shape": True,
    "memorization": True,
    "determinism": True,
    "mdl": True,
    "support": True,
    "robust": True,
    "loo": True,
    "adj": True,
}


@dataclass
class DisambiguationConfig:
    weights: dict[str, float] = field(default_factory=lambda: dict(DEFAULT_WEIGHTS))
    enable: dict[str, bool] = field(default_factory=lambda: dict(DEFAULT_ENABLE))
    max_battery_programs: int = 8
    cluster_theta: float = 0.75          # probe agreement needed to share a cluster
    diversity_bonus: float = 0.15        # attempt_2 bonus for a different SHAPE
    two_pair_equiv_boost: float = 1.25   # 2-pair tasks overfit more -> lean on symmetry
    fallback_soft_boost: float = 3.0     # no verified program: likelihood must dominate priors
    n_mutations: int = 2
    det_runs: int = 3                    # extra repeats of the test input for the RNG check
    max_color_perms: int = 4
    policy: str = "full"                 # full | naive_top2 | first_verified | random
    seed: int = 0

    def weight(self, name: str) -> float:
        if name in self.enable and not self.enable[name]:
            return 0.0
        return float(self.weights.get(name, 0.0))


# ---------------------------------------------------------------------------
# static program analysis (pure, no execution)
# ---------------------------------------------------------------------------


def ast_size(code: str) -> int:
    """AST node count with docstrings stripped — an MDL proxy robust to
    whitespace/comment formatting."""
    try:
        tree = ast.parse(code)
    except SyntaxError:
        return len(code)
    for node in ast.walk(tree):
        if isinstance(node, (ast.FunctionDef, ast.AsyncFunctionDef, ast.ClassDef, ast.Module)):
            body = getattr(node, "body", [])
            if (
                body
                and isinstance(body[0], ast.Expr)
                and isinstance(body[0].value, ast.Constant)
                and isinstance(body[0].value.value, str)
            ):
                node.body = body[1:] or [ast.Pass()]
    n = 0
    for node in ast.walk(tree):
        if isinstance(node, ast.Constant) and isinstance(node.value, str):
            n += max(1, len(node.value) // 32)  # long strings cost a little
        else:
            n += 1
    return n


def memorization_score(code: str, train: Sequence[tuple[Grid, Grid]]) -> float:
    """1.0 when a full train grid appears as a literal; 0.75 when >=2 distinct
    full rows (len>=6) of one train grid appear; else 0.0."""
    try:
        tree = ast.parse(code)
    except SyntaxError:
        return 0.0

    def flatten_const(node) -> Optional[list[int]]:
        """Flatten nested list/tuple integer constants."""
        if isinstance(node, (ast.List, ast.Tuple)):
            out: list[int] = []
            for e in node.elts:
                sub = flatten_const(e)
                if sub is None:
                    return None
                out.extend(sub)
            return out
        if isinstance(node, ast.Constant) and isinstance(node.value, (int, bool)) :
            return [int(node.value)]
        if isinstance(node, ast.UnaryOp) and isinstance(node.op, ast.USub):
            sub = flatten_const(node.operand)
            return [-v for v in sub] if sub is not None else None
        return None

    literal_seqs: list[list[int]] = []
    for node in ast.walk(tree):
        if isinstance(node, (ast.List, ast.Tuple)):
            flat = flatten_const(node)
            if flat and len(flat) >= 4:
                literal_seqs.append(flat)
    code_norm = re.sub(r"\s+", "", code)
    full_hit = 0.0
    row_hits: dict[int, set[tuple[int, ...]]] = {}
    for gi, (x, y) in enumerate(train):
        for g in (x, y):
            a = np.asarray(g)
            flat = a.reshape(-1).tolist()
            if len(flat) >= 4:
                # exact flattened grid as a literal
                if any(seq == flat for seq in literal_seqs):
                    full_hit = 1.0
                # CSV serialization inside a string constant
                csv = ",".join(str(v) for v in flat)
                if len(flat) >= 8 and csv in code_norm:
                    full_hit = 1.0
            if a.shape[1] >= 6:
                for row in a.tolist():
                    key = tuple(row)
                    if any(seq == list(key) for seq in literal_seqs):
                        row_hits.setdefault(gi, set()).add(key)
    if full_hit:
        return 1.0
    if any(len(v) >= 2 for v in row_hits.values()):
        return 0.75
    return 0.0


# ---------------------------------------------------------------------------
# feature computation (sandbox battery — still zero LLM tokens)
# ---------------------------------------------------------------------------


@dataclass
class FeatureBundle:
    """Everything the ranking needs; picklable so eval_local can cache it and
    run ablations offline without touching the sandbox again."""

    task_id: str
    candidate_indices: list[int]          # indices into pool.candidates (battery set)
    test_outputs: list[list[Optional[Grid]]]  # per battery candidate
    soft_mean: list[float]
    verified: list[bool]
    equiv_train: list[float]
    equiv_test: list[float]
    shape_violation: list[float]
    memorization: list[float]
    determinism_ok: list[bool]
    mdl_nodes: list[int]
    robustness: list[float]
    agreement: Optional[np.ndarray]       # pairwise probe agreement, battery set
    kept_syms: list[str]                  # names of gated symmetries (diagnostics)
    loo: list[float] = field(default_factory=list)
    adj: list[float] = field(default_factory=list)
    generation_order: list[int] = field(default_factory=list)  # battery idx -> pool order


def _probe_battery(
    task: Task, cfg: DisambiguationConfig, rng: np.random.Generator
) -> tuple[list[tuple[str, Grid, float]], list[str]]:
    """(label, grid, weight) probes for behavioral clustering, plus symmetry names."""
    probes: list[tuple[str, Grid, float]] = []
    sym_names: list[str] = []
    d4 = [e for e in D4_ELEMENTS]
    perms = plausible_color_perms(task.train, max_perms=cfg.max_color_perms)
    for t, x in enumerate(task.test):
        probes.append((f"test{t}", x, 1.0))
        for e in d4[1:]:
            probes.append((f"test{t}:{d4_name(e)}", apply_d4(x, e), 0.25))
        for pi, perm in enumerate(perms):
            probes.append((f"test{t}:perm{pi}", apply_color_perm(x, perm), 0.25))
        for m in range(cfg.n_mutations):
            probes.append((f"test{t}:mut{m}", perturb_cells(x, rng, 1), 0.25))
        probes.append((f"test{t}:pad", pad_border(x), 0.25))
    return probes, sym_names


def _battery_candidates(pool, task: Task, cfg: DisambiguationConfig) -> list[int]:
    """Pick up to max_battery_programs candidates: one representative (shortest
    code) per verified behavior group first, then extra members, then top
    partials. Preserves pool (generation) order in the returned list."""
    groups = pool.verified_behaviors()
    reps: list[int] = []
    extras: list[int] = []
    for key, members in sorted(groups.items(), key=lambda kv: -len(kv[1])):
        if key == ("incomplete",):
            extras.extend(members)
            continue
        ordered = sorted(members, key=lambda i: ast_size(pool.candidates[i].code))
        reps.append(ordered[0])
        extras.extend(ordered[1:])
    partials = pool.best_partial(n=cfg.max_battery_programs)
    chosen = reps[: cfg.max_battery_programs]
    for i in extras + partials:
        if len(chosen) >= cfg.max_battery_programs:
            break
        if i not in chosen:
            chosen.append(i)
    return sorted(chosen)


def compute_features(
    task: Task,
    pool,
    executor,
    cfg: DisambiguationConfig,
    rng: Optional[np.random.Generator] = None,
    loo_map: Optional[dict] = None,
    adj_votes: Optional[dict] = None,
) -> FeatureBundle:
    """Run the full offline battery. `executor` is a Sandbox (or LocalExecutor).

    loo_map:  {behavior_key_tuple: corroboration in [0,1]} keyed by the group
              keys of ProgramPool.verified_behaviors().
    adj_votes: {behavior_key_tuple: vote count}.
    """
    if rng is None:
        rng = np.random.default_rng(cfg.seed)
    battery = _battery_candidates(pool, task, cfg)
    n = len(battery)
    empty = FeatureBundle(
        task_id=task.task_id, candidate_indices=battery, test_outputs=[], soft_mean=[],
        verified=[], equiv_train=[], equiv_test=[], shape_violation=[], memorization=[],
        determinism_ok=[], mdl_nodes=[], robustness=[], agreement=None, kept_syms=[],
        generation_order=battery,
    )
    if n == 0:
        return empty

    cands = [pool.candidates[i] for i in battery]

    # ---- symmetries: data-level plausibility ------------------------------
    d4_plausible = plausible_d4(task.train)
    perm_plausible = plausible_color_perms(task.train, max_perms=cfg.max_color_perms)

    # ---- phase 1 jobs: train pseudo-labels for ALL plausible syms ---------
    # job layout helper
    jobs: list[tuple[str, Grid]] = []
    job_tags: list[tuple] = []  # (kind, cand_i, sym_i, pair_k)

    syms = [("d4", e) for e in d4_plausible] + [("perm", p) for p in perm_plausible]

    def apply_sym(g: Grid, sym) -> Grid:
        kind, s = sym
        return apply_d4(g, s) if kind == "d4" else apply_color_perm(g, s)

    verified_idx = [i for i in range(n) if cands[i].verified]
    gate_set = verified_idx if verified_idx else list(range(n))
    if not verified_idx:
        # fallback mode: equivariance inactive (see per-candidate features) -
        # skip all pseudo-label jobs to save sandbox time
        syms = []
    for ci in gate_set:
        for si, sym in enumerate(syms):
            for k, (x, y) in enumerate(task.train):
                jobs.append((cands[ci].code, apply_sym(x, sym)))
                job_tags.append(("pseudo", ci, si, k))
    results = executor.run_batch(jobs) if jobs else []

    # pseudo-pass[c][s] = all pairs exact
    pseudo_pass: dict[tuple[int, int], bool] = {}
    pseudo_cell: dict[tuple[int, int], list[bool]] = {}
    for res, tag in zip(results, job_tags):
        _, ci, si, k = tag
        x, y = task.train[k]
        expected = apply_sym(y, syms[si])
        ok = res.ok and grids_equal(res.output, expected)
        pseudo_cell.setdefault((ci, si), []).append(ok)
    for key, cells in pseudo_cell.items():
        pseudo_pass[key] = all(cells)

    # ---- pool-majority resolvability gate ---------------------------------
    # Group gated candidates by test-output behavior; weight = min(3, size).
    beh_groups: dict[tuple, list[int]] = {}
    for ci in gate_set:
        key = tuple(
            grid_key(o) if o is not None else None for o in cands[ci].test_outputs
        )
        beh_groups.setdefault(key, []).append(ci)
    kept_syms_idx: list[int] = []
    if verified_idx:
        total_w = 0.0
        pass_w: dict[int, float] = {si: 0.0 for si in range(len(syms))}
        for key, members in beh_groups.items():
            w = float(min(3, len(members)))
            total_w += w
            for si in range(len(syms)):
                n_pass = sum(1 for ci in members if pseudo_pass.get((ci, si), False))
                if n_pass * 2 >= len(members):
                    pass_w[si] += w
        for si in range(len(syms)):
            if total_w == 0 or pass_w[si] / total_w >= 0.5:
                kept_syms_idx.append(si)
    else:
        kept_syms_idx = list(range(len(syms)))

    kept_names = [
        (d4_name(syms[si][1]) if syms[si][0] == "d4" else f"perm{syms[si][1]}")
        for si in kept_syms_idx
    ]

    # ---- phase 2 jobs: equiv on kept syms (all battery cands), probes,
    #      determinism, plain test outputs -----------------------------------
    jobs2: list[tuple[str, Grid]] = []
    tags2: list[tuple] = []
    probes, _ = _probe_battery(task, cfg, rng)
    for ci in range(n):
        code = cands[ci].code
        for si in kept_syms_idx:
            sym = syms[si]
            if ci not in gate_set or pseudo_pass.get((ci, si)) is None:
                for k, (x, y) in enumerate(task.train):
                    jobs2.append((code, apply_sym(x, sym)))
                    tags2.append(("pseudo", ci, si, k))
            for t, x in enumerate(task.test):
                jobs2.append((code, apply_sym(x, sym)))
                tags2.append(("symtest", ci, si, t))
        for pi, (label, g, w) in enumerate(probes):
            jobs2.append((code, g))
            tags2.append(("probe", ci, pi, 0))
        for t, x in enumerate(task.test):
            for run_k in range(cfg.det_runs):
                jobs2.append((code, x))
                tags2.append(("det", ci, run_k, t))
    results2 = executor.run_batch(jobs2) if jobs2 else []

    # collect all phase-2 outputs in one pass
    symtest_out: dict[tuple[int, int, int], Optional[Grid]] = {}
    probe_out: dict[tuple[int, int], Optional[Grid]] = {}
    det_out: dict[tuple[int, int, int], Optional[Grid]] = {}
    for res, tag in zip(results2, tags2):
        kind, ci, si, k = tag
        out = res.output if res.ok else None
        if kind == "pseudo":
            # deferred pseudo-label runs for non-gate candidates on kept syms
            expected = apply_sym(task.train[k][1], syms[si])
            ok = res.ok and grids_equal(res.output, expected)
            pseudo_cell.setdefault((ci, si), []).append(ok)
            pseudo_pass[(ci, si)] = all(pseudo_cell[(ci, si)])
        elif kind == "symtest":
            symtest_out[(ci, si, k)] = out
        elif kind == "probe":
            probe_out[(ci, si)] = out
        elif kind == "det":
            det_out[(ci, si, k)] = out

    # plain test outputs come from probe index of label test{t}
    test_outputs: list[list[Optional[Grid]]] = []
    for ci in range(n):
        outs = []
        for t in range(len(task.test)):
            pi = next(i for i, (label, _, _) in enumerate(probes) if label == f"test{t}")
            outs.append(probe_out.get((ci, pi)))
        test_outputs.append(outs)

    # ---- per-candidate features -------------------------------------------
    # Equivariance is a generalization REFINER among train-exact programs. In
    # fallback mode (nothing verified) it is disabled: trivially-equivariant
    # programs (identity-like) would otherwise outrank high-soft-score ones,
    # and pool-majority gating is meaningless without verified behaviors.
    equiv_active = bool(verified_idx)
    equiv_train: list[float] = []
    equiv_test: list[float] = []
    determinism_ok: list[bool] = []
    robustness: list[float] = []
    shape_viol: list[float] = []
    memor: list[float] = []
    mdl: list[int] = []
    laws = consistent_shape_laws(task.train)
    for ci in range(n):
        cand = cands[ci]
        # equiv train over kept syms
        if equiv_active:
            vals = [1.0 if pseudo_pass.get((ci, si), False) else 0.0 for si in kept_syms_idx]
            equiv_train.append(float(np.mean(vals)) if vals else 0.0)
        else:
            equiv_train.append(0.0)
        # equiv test: P(s·x_t) == s·P(x_t)
        tt = []
        if equiv_active:
            for si in kept_syms_idx:
                sym = syms[si]
                for t in range(len(task.test)):
                    po = symtest_out.get((ci, si, t))
                    base = test_outputs[ci][t] if t < len(test_outputs[ci]) else None
                    if po is None or base is None:
                        tt.append(0.0)
                        continue
                    expected = apply_sym(base, sym)
                    tt.append(1.0 if grids_equal(po, expected) else 0.0)
        equiv_test.append(float(np.mean(tt)) if tt else 0.0)
        # determinism: repeated test runs identical to the probe-run output
        det_ok = True
        for t in range(len(task.test)):
            base = test_outputs[ci][t] if t < len(test_outputs[ci]) else None
            for run_k in range(cfg.det_runs):
                a = det_out.get((ci, run_k, t))
                if a is not None and base is not None and not grids_equal(a, base):
                    det_ok = False
        determinism_ok.append(det_ok)
        # robustness: fraction of probe runs that produced output
        pruns = [probe_out.get((ci, pi)) is not None for pi in range(len(probes))]
        robustness.append(float(np.mean(pruns)) if pruns else 0.0)
        # shape-law violation fraction over test inputs
        if laws:
            viol = []
            for t in range(len(task.test)):
                o = test_outputs[ci][t] if t < len(test_outputs[ci]) else None
                viol.append(
                    1.0 if shape_law_violation(laws, task.test[t].shape, None if o is None else o.shape) else 0.0
                )
            shape_viol.append(float(np.mean(viol)))
        else:
            shape_viol.append(0.0)
        memor.append(memorization_score(cand.code, task.train))
        mdl.append(ast_size(cand.code))

    # ---- agreement matrix over probes --------------------------------------
    agreement = np.eye(n)
    weights = np.array([w for (_, _, w) in probes])
    for a in range(n):
        for b in range(a + 1, n):
            num = 0.0
            den = 0.0
            for pi in range(len(probes)):
                oa = probe_out.get((a, pi))
                ob = probe_out.get((b, pi))
                if oa is None or ob is None:
                    continue
                den += weights[pi]
                if grids_equal(oa, ob):
                    num += weights[pi]
            agree = num / den if den > 0 else 0.0
            agreement[a, b] = agreement[b, a] = agree

    # ---- LOO / adjudication injection ---------------------------------------
    groups = pool.verified_behaviors()
    loo_vals = [0.0] * n
    adj_vals = [0.0] * n
    if loo_map or adj_votes:
        for ci in range(n):
            idx = battery[ci]
            cand = pool.candidates[idx]
            if not cand.verified or any(o is None for o in cand.test_outputs):
                continue
            key = tuple(grid_key(o) for o in cand.test_outputs)
            if loo_map:
                loo_vals[ci] = float(loo_map.get(key, 0.0))
            if adj_votes:
                adj_vals[ci] = float(adj_votes.get(key, 0.0))

    return FeatureBundle(
        task_id=task.task_id,
        candidate_indices=battery,
        test_outputs=test_outputs,
        soft_mean=[c.soft_mean for c in cands],
        verified=[c.verified for c in cands],
        equiv_train=equiv_train,
        equiv_test=equiv_test,
        shape_violation=shape_viol,
        memorization=memor,
        determinism_ok=determinism_ok,
        mdl_nodes=mdl,
        robustness=robustness,
        agreement=agreement,
        kept_syms=kept_names,
        loo=loo_vals,
        adj=adj_vals,
        generation_order=list(range(n)),
    )


# ---------------------------------------------------------------------------
# ranking + selection (pure — used by ablations)
# ---------------------------------------------------------------------------


def _mdl_norm(nodes: Sequence[int]) -> list[float]:
    if not nodes:
        return []
    lo, hi = min(nodes), max(nodes)
    if hi == lo:
        return [1.0] * len(nodes)
    return [1.0 - (v - lo) / (hi - lo) for v in nodes]


def _cluster(bundle: FeatureBundle, cfg: DisambiguationConfig) -> list[list[int]]:
    """Greedy clustering by probe agreement (threshold cluster_theta)."""
    n = len(bundle.candidate_indices)
    if n == 0:
        return []
    clusters: list[list[int]] = []
    order = sorted(
        range(n),
        key=lambda i: (not bundle.verified[i], -bundle.soft_mean[i], bundle.mdl_nodes[i]),
    )
    A = bundle.agreement if bundle.agreement is not None else np.eye(n)
    for i in order:
        placed = False
        for cl in clusters:
            rep = cl[0]
            if A[i, rep] >= cfg.cluster_theta:
                cl.append(i)
                placed = True
                break
        if not placed:
            clusters.append([i])
    return clusters


def clusters_of(bundle: FeatureBundle, cfg: DisambiguationConfig) -> tuple[list[list[int]], dict[int, int]]:
    clusters = _cluster(bundle, cfg)
    cid_of = {i: c for c, cl in enumerate(clusters) for i in cl}
    return clusters, cid_of


def score_bundle(bundle: FeatureBundle, cfg: DisambiguationConfig, n_pairs: int) -> list[float]:
    n = len(bundle.candidate_indices)
    if n == 0:
        return []
    W = cfg.weight
    mdl_n = _mdl_norm(bundle.mdl_nodes)
    clusters, _ = clusters_of(bundle, cfg)
    support = [0.0] * n
    for cl in clusters:
        for i in cl:
            support[i] = float(len(cl))
    support_norm = [s / n for s in support]

    equiv_boost = cfg.two_pair_equiv_boost if n_pairs <= 2 else 1.0
    any_verified = any(bundle.verified)
    soft_w = W("soft") * (cfg.fallback_soft_boost if not any_verified else 1.0)
    scores: list[float] = []
    for i in range(n):
        s = 0.0
        s += W("verified") * (1.0 if bundle.verified[i] else 0.0)
        s += soft_w * bundle.soft_mean[i]
        if cfg.enable.get("support", True):
            s += W("support") * support_norm[i]
        if cfg.enable.get("equiv", True):
            equiv = 0.6 * bundle.equiv_train[i] + 0.4 * bundle.equiv_test[i]
            s += W("equiv") * equiv * equiv_boost
        if cfg.enable.get("shape", True):
            s -= W("shape") * bundle.shape_violation[i]
        if cfg.enable.get("memorization", True):
            s -= W("memorization") * bundle.memorization[i]
        if cfg.enable.get("determinism", True) and not bundle.determinism_ok[i]:
            s -= W("nondet")
        if cfg.enable.get("mdl", True):
            s += W("mdl") * mdl_n[i]
        if cfg.enable.get("robust", True):
            s += W("robust") * bundle.robustness[i]
        if cfg.enable.get("loo", True):
            s += W("loo") * (bundle.loo[i] if bundle.loo else 0.0)
        if cfg.enable.get("adj", True):
            s += W("adj") * (bundle.adj[i] if bundle.adj else 0.0)
        scores.append(s)
    return scores


def rank_and_select(
    bundle: FeatureBundle,
    cfg: DisambiguationConfig,
    task: Task,
    fallback_outputs: Optional[Sequence[Optional[Grid]]] = None,
) -> tuple[DisambiguationResult, list[tuple[Optional[Grid], Optional[Grid]]]]:
    """Pure ranking + pass@2-aware attempt selection from a FeatureBundle.

    fallback_outputs: per-test-input grids used only when NO candidate produced
    any usable output (schema must always be satisfiable).
    """
    n = len(bundle.candidate_indices)
    notes: list[str] = []
    scores = score_bundle(bundle, cfg, task.n_pairs)
    clusters, cid_of = clusters_of(bundle, cfg)

    cluster_scores: list[ClusterScore] = []
    for cl in clusters:
        best = max(cl, key=lambda i: scores[i])
        outs = bundle.test_outputs[best]
        cluster_scores.append(
            ClusterScore(
                rep_index=best,
                test_output=outs[0] if outs else None,
                score=scores[best],
                support=float(len(cl)),
                members=list(cl),
                features={
                    "soft": bundle.soft_mean[best],
                    "verified": float(bundle.verified[best]),
                    "equiv_train": bundle.equiv_train[best],
                    "equiv_test": bundle.equiv_test[best],
                    "shape_violation": bundle.shape_violation[best],
                    "memorization": bundle.memorization[best],
                    "mdl_nodes": float(bundle.mdl_nodes[best]),
                    "robust": bundle.robustness[best],
                    "loo": bundle.loo[best] if bundle.loo else 0.0,
                    "adj": bundle.adj[best] if bundle.adj else 0.0,
                },
            )
        )
    cluster_scores.sort(key=lambda c: -c.score)

    n_test = task.n_test
    attempts: list[tuple[Optional[Grid], Optional[Grid]]] = []

    if cfg.policy == "random" and n:
        rng = np.random.default_rng(cfg.seed)
        perm = list(rng.permutation(n))
        for t in range(n_test):
            outs = [bundle.test_outputs[i][t] for i in perm if t < len(bundle.test_outputs[i])]
            outs = [o for o in outs if o is not None]
            a1 = outs[0] if outs else None
            a2 = next((o for o in outs[1:] if not grids_equal(o, a1)), a1)
            attempts.append((a1, a2))
    elif cfg.policy == "first_verified":
        # Poetiq-style baseline: first verified program in generation order wins.
        order = sorted(range(n), key=lambda i: (not bundle.verified[i], bundle.generation_order[i]))
        for t in range(n_test):
            outs = [
                bundle.test_outputs[i][t]
                for i in order
                if t < len(bundle.test_outputs[i]) and bundle.test_outputs[i][t] is not None
            ]
            a1 = outs[0] if outs else None
            a2 = next((o for o in outs[1:] if not grids_equal(o, a1)), a1)
            attempts.append((a1, a2))
    else:
        for t in range(n_test):
            ranked_outs: list[tuple[float, Optional[Grid], tuple]] = []
            for cs in cluster_scores:
                # best member of this cluster that actually produced output for test t
                o = None
                sc = cs.score
                for m in sorted(cs.members, key=lambda i: -scores[i]):
                    cand_o = bundle.test_outputs[m][t] if t < len(bundle.test_outputs[m]) else None
                    if cand_o is not None:
                        o = cand_o
                        sc = scores[m]
                        break
                if o is not None:
                    ranked_outs.append((sc, o, o.shape))
            if not ranked_outs:
                for i in sorted(range(n), key=lambda i: -scores[i]):
                    o = bundle.test_outputs[i][t] if t < len(bundle.test_outputs[i]) else None
                    if o is not None:
                        ranked_outs.append((scores[i], o, o.shape))
            if not ranked_outs:
                fb = fallback_outputs[t] if fallback_outputs and t < len(fallback_outputs) else None
                attempts.append((fb, fb))
                notes.append(f"test{t}: no candidate output; fallback grid used")
                continue
            a1_score, a1, a1_shape = ranked_outs[0]
            # attempt_2: best DIFFERENT grid; +diversity bonus when shape differs
            a2 = None
            if cfg.policy == "naive_top2":
                for sc, o, shp in ranked_outs[1:]:
                    if not grids_equal(o, a1):
                        a2 = o
                        break
            else:
                best2 = None
                best2_val = None
                for sc, o, shp in ranked_outs[1:]:
                    if grids_equal(o, a1):
                        continue
                    val = sc + (cfg.diversity_bonus if shp != a1_shape else 0.0)
                    if best2_val is None or val > best2_val:
                        best2_val, best2 = val, o
                a2 = best2
            if a2 is None:
                a2 = a1
                notes.append(f"test{t}: only one distinct output; attempt_2 duplicates attempt_1")
            attempts.append((a1, a2))

    features_out = []
    for i in range(n):
        features_out.append(
            {
                "pool_index": bundle.candidate_indices[i],
                "score": scores[i] if i < len(scores) else 0.0,
                "cluster": cid_of.get(i, -1),
                "verified": bundle.verified[i],
                "soft_mean": bundle.soft_mean[i],
                "equiv_train": bundle.equiv_train[i],
                "equiv_test": bundle.equiv_test[i],
                "shape_violation": bundle.shape_violation[i],
                "memorization": bundle.memorization[i],
                "determinism_ok": bundle.determinism_ok[i],
                "mdl_nodes": bundle.mdl_nodes[i],
                "robust": bundle.robustness[i],
                "loo": bundle.loo[i] if bundle.loo else 0.0,
                "adj": bundle.adj[i] if bundle.adj else 0.0,
            }
        )

    return DisambiguationResult(
        attempt_1=attempts[0][0] if attempts else None,
        attempt_2=attempts[0][1] if attempts else None,
        clusters=cluster_scores,
        features=features_out,
        agreement=bundle.agreement,
        notes=notes,
    ), attempts


def disambiguate(
    task: Task,
    pool,
    executor,
    cfg: Optional[DisambiguationConfig] = None,
    rng: Optional[np.random.Generator] = None,
    loo_map: Optional[dict] = None,
    adj_votes: Optional[dict] = None,
    fallback_outputs: Optional[Sequence[Optional[Grid]]] = None,
) -> tuple[DisambiguationResult, list[tuple[Optional[Grid], Optional[Grid]]], FeatureBundle]:
    """Full pipeline: features (sandbox battery) -> rank -> select."""
    cfg = cfg or DisambiguationConfig()
    bundle = compute_features(task, pool, executor, cfg, rng=rng, loo_map=loo_map, adj_votes=adj_votes)
    result, attempts = rank_and_select(bundle, cfg, task, fallback_outputs=fallback_outputs)
    return result, attempts, bundle


In [ ]:
%%writefile /kaggle/working/h3/harness/prompts.py
"""Prompt construction for program synthesis.

Adapted from the Poetiq solver prompts (MIT License, Copyright 2025 Poetiq,
Inc. — see reference/poetiq-solver/LICENSE.txt), with three deliberate changes:
  1. grids rendered as CSV (`0,1,2`) — benchmarked best of 9 formats (Land);
  2. explicit anti-overfit contract (no memorized grids — the harness screens
     for literals; no hardcoded positions/colors/sizes; must generalize to
     unseen sizes) — this is the ARC-AGI-2 "less brute-forcible" response;
  3. extra compact structural analysis per grid (connected components,
     palettes) — the Gre24 finding that extra text representations help.

Land's documented negative result — "prescriptive prompting templates
systematically reduce hypothesis diversity and degrade performance" — is why
we keep TWO solver prompt variants of different verbosity and rotate them
across experts/iterations instead of one heavily-engineered template.
"""
from __future__ import annotations

from typing import Optional, Sequence

import numpy as np

from .grid import analysis_block, format_csv
from .types import Grid, Task

# ---------------------------------------------------------------------------
# solver prompts (variant A: fuller guidance; variant B: minimal — Land found
# minimal prompts preserve hypothesis diversity)
# ---------------------------------------------------------------------------

SOLVER_PROMPT_A = '''You are an expert at solving ARC-AGI abstract reasoning tasks by writing Python programs, and a master Python coder.

Analyze the input/output examples below, infer the SINGLE general transformation rule consistent with ALL of them, and implement it as a function `transform`.

Think about:
- Objects (connected components), their colors, sizes, positions, and relationships.
- Operations: moving, rotating, reflecting, scaling, cropping, recoloring, counting, completing patterns, gravity, tiling, drawing lines/rays, enclosing, symmetry.
- Whether the rule composes several steps, and whether some step is gated by a contextual cue (e.g. a color or position decides which branch applies).
- Whether the rule treats colors by ROLE (e.g. "the unique object", "the largest component", "the minority color") rather than by identity.

GENERALIZATION CONTRACT (critical — this is ARC-AGI-2, the test input differs from the training ones):
- Your program will be run on a NEW input grid, possibly of a different size, with objects in different positions, and possibly colors that never appear in the training examples. It must not crash and must apply the same rule.
- NEVER hardcode: no memorized grids, no absolute cell coordinates, no fixed grid sizes, no per-example special cases. Programs containing literal copies of training grids are automatically rejected.
- Prefer role-based color logic over color identity when the examples suggest it (e.g. "the least common color", "the color of the object touching the border").
- The rule should be geometrically natural: if the examples do not force a direction (up/down/left/right) or a specific color, your program should behave consistently under rotation, reflection, and color renaming.
- Output shape: derive it from the rule (same size, cropped, scaled, constant...), never assume.

IMPLEMENTATION:
- Signature: `def transform(grid: np.ndarray) -> np.ndarray`. `grid` is a 2-D int array (values 0-9). Return a NEW 2-D int array; do not modify the input in place.
- Available: numpy, cv2, Python standard library. Deterministic and pure: no randomness, no I/O, no global state.
- Handle any input size >= 1x1 gracefully.

OUTPUT FORMAT:
- A brief explanation, then the complete program in a SINGLE markdown ```python code block. No `__main__` block, no test code, no calls to transform.

**PROBLEM:**

$$problem$$
'''

SOLVER_PROMPT_B = '''Below is an ARC task: training input/output grid pairs (CSV) and test input(s). Write a Python function `transform(grid: np.ndarray) -> np.ndarray` (numpy available, deterministic, no hardcoded grids/positions/sizes — it must generalize to the test input, which may have a different size and unseen colors) that maps every input to its output according to the single rule consistent with all training pairs.

Reply with a short explanation and then exactly one markdown ```python code block containing the program (no `__main__`, no test calls).

$$problem$$
'''

FEEDBACK_PROMPT = '''**PREVIOUS ATTEMPTS:**

Below are the best programs so far. None is fully correct. Each shows its code, its execution feedback on the training examples (mismatching cells are shown as `predicted/expected`), and a score in [0, 1]. Study the failure patterns, form a better hypothesis, and produce a NEW program that fixes all the issues. Do not merely patch a failing example — find the rule that explains ALL examples.

$$feedback$$
'''

CONSISTENCY_NUDGE = '''**NOTE:** Your previous program passed ALL training pairs. It was kept. The training pairs are necessary but NOT sufficient evidence: the test input may differ in size, positions, or colors. Re-examine whether your rule is stated by role rather than by accident of the training grids (absolute positions, specific color identities, literal sizes). Now propose either (a) the same program if you are confident it generalizes, or (b) a DIFFERENT hypothesis that also explains all training pairs but generalizes better. Always output one ```python code block.
'''

ADJUDICATION_PROMPT = '''Two or more Python programs below ALL reproduce every training input/output pair exactly, yet they produce DIFFERENT outputs on the test input. At most one captures the intended rule.

Decide which program generalizes correctly, and write the final correct program. Reason about:
- Which rule is more natural/general (role-based colors, geometric consistency, no magic constants)?
- Which one would still be consistent if the training grids were rotated, reflected, or recolored?
- Does one of them hardcode accidents of the training set (positions, sizes, color identities)?
- The test input's specifics: sizes, colors, object counts that never appeared in training.

Programs and their test outputs:

$$candidates$$

Output a short justification, then the FINAL program (signature `def transform(grid: np.ndarray) -> np.ndarray`, numpy/cv2/stdlib, deterministic, no memorized grids) in a single markdown ```python code block.

$$problem$$
'''

LOO_PROMPT = '''You are an expert at solving ARC-AGI tasks by writing Python programs.

Below are SOME training pairs of an ARC task (one pair is withheld). Infer the general rule and implement `def transform(grid: np.ndarray) -> np.ndarray` (numpy available; deterministic; no hardcoding of grids, positions, sizes, or color identities — describe the rule by roles so that it applies to ANY pair from this task).

Reply with a short explanation, then exactly one markdown ```python code block.

$$problem$$
'''


# ---------------------------------------------------------------------------
# problem rendering
# ---------------------------------------------------------------------------


def render_task(
    task: Task,
    shuffle: bool = True,
    seed: int = 0,
    omit_pair: Optional[int] = None,
    include_test: bool = True,
    include_analysis: bool = True,
) -> str:
    """CSV rendering of train pairs (+ test inputs), optionally shuffled with
    `seed` (Poetiq's free augmentation: re-ordering pairs every iteration) and
    optionally omitting one pair (LOO folds)."""
    pairs = list(task.train)
    idxs = list(range(len(pairs)))
    if omit_pair is not None:
        idxs = [i for i in idxs if i != omit_pair]
    if shuffle and len(idxs) > 1:
        rng = np.random.default_rng(seed)
        idxs = [idxs[i] for i in rng.permutation(len(idxs))]

    parts = []
    for n, i in enumerate(idxs, start=1):
        x, y = pairs[i]
        parts.append(f"Example #{n}\nInput:\n{format_csv(x)}\n\nOutput:\n{format_csv(y)}")
    if include_test:
        for n, x in enumerate(task.test, start=1):
            parts.append(f"Test input #{n} (produce the output grid for this one):\n{format_csv(x)}")
    text = "\n\n".join(parts)
    if include_analysis:
        shown_pairs = [pairs[i] for i in idxs]
        tests = list(task.test) if include_test else []
        text += "\n\nStructural analysis:\n" + analysis_block(shown_pairs, tests)
    return text


def _fill(template: str, **fields: str) -> str:
    out = template
    for k, v in fields.items():
        out = out.replace(f"$${k}$$", v)
    return out


def build_solver_messages(
    task: Task,
    iteration: int,
    variant: str = "A",
    partials: Optional[Sequence[tuple[str, str, float]]] = None,
    max_partials: int = 5,
    improving_order: bool = True,
    shuffle_seed: int = 0,
) -> list[dict]:
    """Chat messages for one synthesis call.

    partials: (code, feedback_text, soft_score) of previous attempts.
    """
    template = SOLVER_PROMPT_A if variant == "A" else SOLVER_PROMPT_B
    problem = render_task(task, shuffle=True, seed=shuffle_seed + iteration)
    content = _fill(template, problem=problem)
    if partials:
        ranked = sorted(partials, key=lambda p: p[2])
        if not improving_order:
            ranked = ranked[::-1]
        ranked = ranked[-max_partials:]
        blocks = []
        for k, (code, fb, score) in enumerate(ranked, start=1):
            blocks.append(
                f"<solution_{k}>\n<score>{score:.2f}</score>\n<feedback>\n{fb}\n</feedback>\n"
                f"<code>\n```python\n{code}\n```\n</code>\n</solution_{k}>"
            )
        content += "\n\n" + _fill(FEEDBACK_PROMPT, feedback="\n\n".join(blocks))
    return [{"role": "user", "content": content}]


def build_verified_nudge_messages(
    task: Task, verified_code: str, iteration: int, seed: int = 0
) -> list[dict]:
    """When the pool already contains a verified program, later iterations ask
    for a DIVERSE alternative hypothesis (pool diversity feeds disambiguation)."""
    problem = render_task(task, shuffle=True, seed=seed + iteration)
    content = _fill(SOLVER_PROMPT_A, problem=problem)
    content += (
        "\n\n**A PROGRAM THAT PASSES ALL TRAINING PAIRS:**\n```python\n"
        + verified_code
        + "\n```\n\n"
        + CONSISTENCY_NUDGE
    )
    return [{"role": "user", "content": content}]


def build_adjudication_messages(
    task: Task, entries: Sequence[tuple[str, Optional[Grid]]], variant_seed: int = 0
) -> list[dict]:
    """entries: (code, test_output_of_first_test_input) of the disagreeing
    verified representatives."""
    from .grid import format_csv as _fc

    blocks = []
    for k, (code, out) in enumerate(entries, start=1):
        out_txt = _fc(out) if out is not None else "(program crashed on the test input)"
        blocks.append(
            f"<program_{k}>\n```python\n{code}\n```\n"
            f"<its_output_on_test_input_1>\n{out_txt}\n</its_output_on_test_input_1>\n</program_{k}>"
        )
    problem = render_task(task, shuffle=False, include_analysis=False)
    content = _fill(ADJUDICATION_PROMPT, candidates="\n\n".join(blocks), problem=problem)
    return [{"role": "user", "content": content}]


def build_loo_messages(task: Task, omit_pair: int, seed: int = 0) -> list[dict]:
    problem = render_task(task, shuffle=False, seed=seed, omit_pair=omit_pair, include_test=False)
    content = _fill(LOO_PROMPT, problem=problem)
    return [{"role": "user", "content": content}]


In [ ]:
%%writefile /kaggle/working/h3/harness/llm.py
"""LLM clients: injectable interface + vLLM (OpenAI-compatible) + MockLLM.

Everything above this module is LLM-agnostic, which is what makes the entire
disambiguation layer testable offline (no GPU): tests and `eval_local --mock`
inject MockLLM; the Kaggle notebook injects VLLMClient pointed at a local
vLLM server.

vLLM traps handled here (verified upstream):
- Qwen3.8 `reasoning_effort` accepts ONLY {xhigh, medium, low}; passing "high"
  raises HTTP 500 (QwenLM/Qwen3.8#217). Default is xhigh — we always pin
  low/medium explicitly to stay inside the token budget.
- Structured output is unreliable in thinking mode (vLLM#35700) -> we never
  request JSON schema; we extract markdown code blocks tolerantly instead.
"""
from __future__ import annotations

import json
import time
import urllib.error
import urllib.request
from dataclasses import dataclass, field
from typing import Callable, Optional, Protocol

from .types import LLMResponse


class LLMClient(Protocol):
    def chat(
        self,
        messages: list[dict],
        *,
        temperature: float = 1.0,
        max_tokens: int = 4096,
        reasoning_effort: str = "low",
        seed: Optional[int] = None,
    ) -> LLMResponse: ...


# ---------------------------------------------------------------------------
# vLLM / OpenAI-compatible client (stdlib only — works on Kaggle & locally)
# ---------------------------------------------------------------------------


class VLLMClient:
    def __init__(
        self,
        base_url: str = "http://127.0.0.1:8000/v1",
        model: str = "qwen3.8-27b",
        request_timeout: float = 900.0,
        max_retries: int = 3,
    ):
        self.base_url = base_url.rstrip("/")
        self.model = model
        self.request_timeout = request_timeout
        self.max_retries = max_retries
        self.total_prompt_tokens = 0
        self.total_completion_tokens = 0

    def chat(
        self,
        messages: list[dict],
        *,
        temperature: float = 1.0,
        max_tokens: int = 4096,
        reasoning_effort: str = "low",
        seed: Optional[int] = None,
    ) -> LLMResponse:
        assert reasoning_effort in ("low", "medium", "xhigh"), (
            "Qwen3.8 reasoning_effort accepts only low/medium/xhigh ('high' -> HTTP 500)"
        )
        payload = {
            "model": self.model,
            "messages": messages,
            "temperature": temperature,
            "max_tokens": max_tokens,
            "chat_template_kwargs": {
                "enable_thinking": True,
                "reasoning_effort": reasoning_effort,
            },
        }
        if seed is not None:
            payload["seed"] = seed
        data = json.dumps(payload).encode()
        last_err: Optional[Exception] = None
        for attempt in range(self.max_retries):
            try:
                req = urllib.request.Request(
                    f"{self.base_url}/chat/completions",
                    data=data,
                    headers={"Content-Type": "application/json"},
                )
                with urllib.request.urlopen(req, timeout=self.request_timeout) as resp:
                    body = json.loads(resp.read().decode())
                text = body["choices"][0]["message"].get("content") or ""
                reasoning = body["choices"][0]["message"].get("reasoning_content") or ""
                usage = body.get("usage", {})
                pt = int(usage.get("prompt_tokens", 0))
                ct = int(usage.get("completion_tokens", 0))
                self.total_prompt_tokens += pt
                self.total_completion_tokens += ct
                # reasoning_content arrives as a separate field with the qwen3
                # parser; keep it out of `text` (we only parse code blocks).
                del reasoning
                return LLMResponse(text=text, prompt_tokens=pt, completion_tokens=ct, model=self.model)
            except (urllib.error.URLError, urllib.error.HTTPError, TimeoutError, OSError, KeyError) as e:
                last_err = e
                time.sleep(min(2 ** attempt * 2.0, 30.0))
        raise RuntimeError(f"LLM request failed after {self.max_retries} retries: {last_err}")


# ---------------------------------------------------------------------------
# MockLLM
# ---------------------------------------------------------------------------


class MockLLM:
    """Scriptable stand-in for a real LLM. `responder(messages, call_index)`
    returns the raw assistant text; token counts are approximated as len//4 so
    budget accounting is exercised in tests."""

    def __init__(self, responder: Callable[[list[dict], int], str], model: str = "mock"):
        self.responder = responder
        self.model = model
        self.calls = 0
        self.total_prompt_tokens = 0
        self.total_completion_tokens = 0

    def chat(
        self,
        messages: list[dict],
        *,
        temperature: float = 1.0,
        max_tokens: int = 4096,
        reasoning_effort: str = "low",
        seed: Optional[int] = None,
    ) -> LLMResponse:
        text = self.responder(messages, self.calls)
        self.calls += 1
        pt = sum(len(m.get("content", "")) for m in messages) // 4
        ct = len(text) // 4
        self.total_prompt_tokens += pt
        self.total_completion_tokens += ct
        return LLMResponse(text=text, prompt_tokens=pt, completion_tokens=ct, model=self.model)


def scripted_responder(scripts: list[str]) -> Callable[[list[dict], int], str]:
    """Returns responses in order; repeats the last one when exhausted."""

    def responder(messages: list[dict], call_index: int) -> str:
        if not scripts:
            return "no script"
        return scripts[min(call_index, len(scripts) - 1)]

    return responder


# A small library of generic programs used by LibraryMockLLM / --mock runs and
# as the CPU-only fallback inside the Kaggle notebook. These exercise the full
# oracle+disambiguation path; on hard ARC-AGI-2 tasks most fail verification,
# which is exactly the fallback path we must validate.
TEMPLATE_PROGRAMS: list[tuple[str, str]] = [
    (
        "identity",
        """import numpy as np

def transform(grid: np.ndarray) -> np.ndarray:
    return grid.copy()
""",
    ),
    (
        "rot90",
        """import numpy as np

def transform(grid: np.ndarray) -> np.ndarray:
    return np.ascontiguousarray(np.rot90(grid, 1))
""",
    ),
    (
        "rot180",
        """import numpy as np

def transform(grid: np.ndarray) -> np.ndarray:
    return np.ascontiguousarray(np.rot90(grid, 2))
""",
    ),
    (
        "rot270",
        """import numpy as np

def transform(grid: np.ndarray) -> np.ndarray:
    return np.ascontiguousarray(np.rot90(grid, 3))
""",
    ),
    (
        "fliplr",
        """import numpy as np

def transform(grid: np.ndarray) -> np.ndarray:
    return np.ascontiguousarray(grid[:, ::-1])
""",
    ),
    (
        "flipud",
        """import numpy as np

def transform(grid: np.ndarray) -> np.ndarray:
    return np.ascontiguousarray(grid[::-1, :])
""",
    ),
    (
        "transpose",
        """import numpy as np

def transform(grid: np.ndarray) -> np.ndarray:
    return np.ascontiguousarray(grid.T)
""",
    ),
    (
        "top_half",
        """import numpy as np

def transform(grid: np.ndarray) -> np.ndarray:
    return grid[: grid.shape[0] // 2, :].copy()
""",
    ),
    (
        "left_half",
        """import numpy as np

def transform(grid: np.ndarray) -> np.ndarray:
    return grid[:, : grid.shape[1] // 2].copy()
""",
    ),
    (
        "zoom2x",
        """import numpy as np

def transform(grid: np.ndarray) -> np.ndarray:
    return np.kron(grid, np.ones((2, 2), dtype=grid.dtype))
""",
    ),
    (
        "bg_to_zero",
        """import numpy as np

def transform(grid: np.ndarray) -> np.ndarray:
    vals, counts = np.unique(grid, return_counts=True)
    bg = int(vals[np.argmax(counts)])
    out = grid.copy()
    out[out == bg] = 0
    return out
""",
    ),
    (
        "least_to_most",
        """import numpy as np

def transform(grid: np.ndarray) -> np.ndarray:
    vals, counts = np.unique(grid, return_counts=True)
    order = np.argsort(counts)
    out = grid.copy()
    out[out == int(vals[order[0]])] = int(vals[order[-1]])
    return out
""",
    ),
    (
        "keep_largest_component",
        """import numpy as np
from collections import deque

def transform(grid: np.ndarray) -> np.ndarray:
    g = np.asarray(grid)
    h, w = g.shape
    vals, counts = np.unique(g, return_counts=True)
    bg = int(vals[np.argmax(counts)])
    seen = np.zeros_like(g, dtype=bool)
    best, best_size = None, 0
    for i in range(h):
        for j in range(w):
            if seen[i, j] or g[i, j] == bg:
                continue
            color = int(g[i, j])
            q = deque([(i, j)])
            seen[i, j] = True
            cells = []
            while q:
                r, c = q.popleft()
                cells.append((r, c))
                for dr, dc in ((1, 0), (-1, 0), (0, 1), (0, -1)):
                    nr, nc = r + dr, c + dc
                    if 0 <= nr < h and 0 <= nc < w and not seen[nr, nc] and g[nr, nc] == color:
                        seen[nr, nc] = True
                        q.append((nr, nc))
            if len(cells) > best_size:
                best_size, best = len(cells), cells
    out = np.full_like(g, bg)
    if best:
        for r, c in best:
            out[r, c] = g[r, c]
    return out
""",
    ),
    (
        "frame_border",
        """import numpy as np

def transform(grid: np.ndarray) -> np.ndarray:
    g = np.asarray(grid)
    out = np.zeros_like(g)
    out[0, :] = g[0, :]
    out[-1, :] = g[-1, :]
    out[:, 0] = g[:, 0]
    out[:, -1] = g[:, -1]
    return out
""",
    ),
    (
        "gravity_down",
        """import numpy as np

def transform(grid: np.ndarray) -> np.ndarray:
    g = np.asarray(grid)
    vals, counts = np.unique(g, return_counts=True)
    bg = int(vals[np.argmax(counts)])
    out = np.full_like(g, bg)
    for c in range(g.shape[1]):
        col = [v for v in g[:, c] if v != bg]
        for k, v in enumerate(reversed(col)):
            out[g.shape[0] - 1 - k, c] = v
    return out
""",
    ),
    (
        "crop_to_content",
        """import numpy as np

def transform(grid: np.ndarray) -> np.ndarray:
    g = np.asarray(grid)
    vals, counts = np.unique(g, return_counts=True)
    bg = int(vals[np.argmax(counts)])
    mask = g != bg
    if not mask.any():
        return g[:1, :1].copy()
    rows = np.where(mask.any(axis=1))[0]
    cols = np.where(mask.any(axis=0))[0]
    return g[rows[0] : rows[-1] + 1, cols[0] : cols[-1] + 1].copy()
""",
    ),
]


def library_responder(order: Optional[list[int]] = None) -> Callable[[list[dict], int], str]:
    """Cycles through TEMPLATE_PROGRAMS (optionally in a custom order)."""
    idxs = order if order is not None else list(range(len(TEMPLATE_PROGRAMS)))

    def responder(messages: list[dict], call_index: int) -> str:
        name, code = TEMPLATE_PROGRAMS[idxs[call_index % len(idxs)]]
        return f"The rule looks like `{name}`.\n\n```python\n{code}```\n"

    return responder


@dataclass
class MockConfig:
    responder: Callable[[list[dict], int], str] = field(default_factory=library_responder)


In [ ]:
%%writefile /kaggle/working/h3/harness/budget.py
"""Budget accounting (tokens + wall clock) and the hard-stop watchdog."""
from __future__ import annotations

import threading
import time
from dataclasses import dataclass, field


@dataclass
class BudgetStats:
    prompt_tokens: int = 0
    completion_tokens: int = 0
    llm_calls: int = 0
    tasks_done: int = 0

    @property
    def total_tokens(self) -> int:
        return self.prompt_tokens + self.completion_tokens


class Budget:
    """Thread-safe token + wall-clock accountant.

    Per-task token caps implement the quantitative plan in HYPOTHESIS.md:
    ~55-75K raw tokens/task (mostly prefix-cached prompt) and <= ~14K
    completion tokens/task keeps 240 tasks inside the 12h L4x4 envelope.
    """

    def __init__(
        self,
        deadline_ts: float,
        per_task_tokens: int = 75_000,
        total_tokens: int = 24_000_000,
        reserve_seconds: float = 420.0,
    ):
        self.deadline_ts = deadline_ts
        self.per_task_tokens = per_task_tokens
        self.total_tokens = total_tokens
        self.reserve_seconds = reserve_seconds
        self.stats = BudgetStats()
        self._lock = threading.Lock()
        self._task_spent: dict[str, int] = {}

    # -- charging ------------------------------------------------------------

    def charge(self, task_id: str, prompt_tokens: int, completion_tokens: int) -> None:
        with self._lock:
            self.stats.prompt_tokens += prompt_tokens
            self.stats.completion_tokens += completion_tokens
            self.stats.llm_calls += 1
            self._task_spent[task_id] = self._task_spent.get(task_id, 0) + (
                prompt_tokens + completion_tokens
            )

    def task_spent(self, task_id: str) -> int:
        with self._lock:
            return self._task_spent.get(task_id, 0)

    def task_exhausted(self, task_id: str) -> bool:
        return self.task_spent(task_id) >= self.per_task_tokens

    def mark_task_done(self) -> None:
        with self._lock:
            self.stats.tasks_done += 1

    # -- gating ---------------------------------------------------------------

    def expired(self) -> bool:
        return time.time() >= self.deadline_ts - self.reserve_seconds

    def seconds_left(self) -> float:
        return max(0.0, self.deadline_ts - self.reserve_seconds - time.time())

    def can_call(self, task_id: str, est_tokens: int = 0) -> bool:
        with self._lock:
            if self.stats.total_tokens + est_tokens > self.total_tokens:
                return False
            if self._task_spent.get(task_id, 0) + est_tokens > self.per_task_tokens:
                return False
        return not self.expired()

    def per_task_time_cap(self, tasks_remaining: int) -> float:
        """Fair share of remaining wall clock, x1.5 slack, floored at 60s."""
        if tasks_remaining <= 0:
            return 60.0
        return max(60.0, self.seconds_left() / tasks_remaining * 1.5)

    def snapshot(self) -> dict:
        with self._lock:
            return {
                "prompt_tokens": self.stats.prompt_tokens,
                "completion_tokens": self.stats.completion_tokens,
                "llm_calls": self.stats.llm_calls,
                "tasks_done": self.stats.tasks_done,
                "seconds_left": round(self.seconds_left(), 1),
            }


class Watchdog:
    """Hard guarantee that the submission file is finalized before the wall.

    `on_expire` runs in a daemon thread at `deadline_ts`; after
    `hard_kill_grace_s` more the process is terminated with os._exit so a
    wedged main loop cannot eat the submission (lb3389 pattern).
    """

    def __init__(self, deadline_ts: float, on_expire, hard_kill_grace_s: float = 120.0):
        self.deadline_ts = deadline_ts
        self.on_expire = on_expire
        self.hard_kill_grace_s = hard_kill_grace_s
        self.fired = False
        self._timer: threading.Timer | None = None
        self._killer: threading.Timer | None = None

    def start(self) -> None:
        delay = max(0.0, self.deadline_ts - time.time())
        self._timer = threading.Timer(delay, self._fire)
        self._timer.daemon = True
        self._timer.start()

    def _fire(self) -> None:
        self.fired = True
        try:
            self.on_expire()
        finally:
            import os

            self._killer = threading.Timer(self.hard_kill_grace_s, lambda: os._exit(0))
            self._killer.daemon = True
            self._killer.start()

    def cancel(self) -> None:
        if self._timer:
            self._timer.cancel()
        if self._killer:
            self._killer.cancel()


In [ ]:
%%writefile /kaggle/working/h3/harness/submission.py
"""Cumulative submission writer + schema validation.

Competition schema (ARC Prize 2026):
    {task_id: [{"attempt_1": grid, "attempt_2": grid}, ...one per test input...]}
- every task_id must appear; both attempts always present; order matches the
  test-input order; grids are rectangular int matrices.

The file is rewritten atomically (tmp + os.replace) after EVERY task so a
12h hard-kill always leaves a complete, valid submission behind. Unfinished
tasks keep heuristic fallbacks: attempt_1 = the test input itself (identity
guess — correct for a nonzero slice of ARC tasks), attempt_2 = background fill.
"""
from __future__ import annotations

import json
import os
import threading
from pathlib import Path
from typing import Optional, Sequence

import numpy as np

from .grid import background_color
from .types import Grid, Task


def fallback_attempts(test_input: Grid) -> list[dict]:
    x = np.asarray(test_input)
    a1 = x.tolist()
    bg = background_color(x)
    a2 = np.full_like(x, bg).tolist()
    return [{"attempt_1": a1, "attempt_2": a2}]


def _grid_to_list(g: Optional[Grid], like: Grid) -> list:
    if g is None:
        return np.zeros_like(np.asarray(like)).tolist()
    a = np.asarray(g)
    return [[int(v) for v in row] for row in a]


class SubmissionWriter:
    def __init__(self, path: str | Path, tasks: Sequence[Task]):
        self.path = Path(path)
        self.tasks = {t.task_id: t for t in tasks}
        self._lock = threading.RLock()
        self._data: dict[str, list[dict]] = {}
        for t in tasks:
            self._data[t.task_id] = [fallback_attempts(x)[0] for x in t.test]
        self.flush()

    def update(
        self,
        task_id: str,
        attempts: Sequence[tuple[Optional[Grid], Optional[Grid]]],
    ) -> None:
        task = self.tasks.get(task_id)
        if task is None:
            raise KeyError(f"unknown task_id {task_id}")
        packs = []
        for t_idx, x in enumerate(task.test):
            if t_idx < len(attempts):
                a1, a2 = attempts[t_idx]
            else:
                a1, a2 = None, None
            g1 = _grid_to_list(a1, x)
            g2 = _grid_to_list(a2, x)
            packs.append({"attempt_1": g1, "attempt_2": g2})
        with self._lock:
            self._data[task_id] = packs
            self.flush()

    def flush(self) -> None:
        with self._lock:
            tmp = self.path.with_suffix(".tmp")
            self.path.parent.mkdir(parents=True, exist_ok=True)
            with open(tmp, "w", encoding="utf-8") as f:
                json.dump(self._data, f)
            os.replace(tmp, self.path)

    def data(self) -> dict:
        with self._lock:
            return json.loads(json.dumps(self._data))


def validate_submission(data: dict, tasks: Sequence[Task]) -> list[str]:
    """Return a list of schema problems (empty = valid)."""
    problems: list[str] = []
    for t in tasks:
        if t.task_id not in data:
            problems.append(f"{t.task_id}: missing")
            continue
        packs = data[t.task_id]
        if not isinstance(packs, list) or len(packs) != t.n_test:
            problems.append(f"{t.task_id}: expected {t.n_test} packs, got {len(packs) if isinstance(packs, list) else '?'}")
            continue
        for i, pack in enumerate(packs):
            for key in ("attempt_1", "attempt_2"):
                g = pack.get(key)
                if not isinstance(g, list) or not g:
                    problems.append(f"{t.task_id}[{i}].{key}: missing/empty")
                    continue
                widths = {len(r) for r in g if isinstance(r, list)}
                if len(widths) != 1 or not all(
                    isinstance(v, int) and 0 <= v <= 9 for r in g for v in r
                ):
                    problems.append(f"{t.task_id}[{i}].{key}: non-rectangular or bad values")
    return problems


def score_submission(data: dict, tasks: Sequence[Task], solutions: dict[str, list]) -> dict:
    """pass@2 ground-truth scoring: per test output and mean over all outputs."""
    total = 0
    hits = 0
    per_task: dict[str, float] = {}
    for t in tasks:
        gt = solutions.get(t.task_id)
        if gt is None:
            continue
        packs = data.get(t.task_id, [])
        task_hits = 0
        for i, truth in enumerate(gt):
            total += 1
            if i >= len(packs):
                continue
            pack = packs[i]
            if pack.get("attempt_1") == truth or pack.get("attempt_2") == truth:
                hits += 1
                task_hits += 1
        per_task[t.task_id] = task_hits / max(1, len(gt))
    return {
        "outputs_total": total,
        "outputs_hit": hits,
        "score": hits / total if total else 0.0,
        "per_task": per_task,
    }


In [ ]:
%%writefile /kaggle/working/h3/harness/data.py
"""Loading ARC challenge/solution JSON files into Task objects."""
from __future__ import annotations

import json
from pathlib import Path
from typing import Optional

import numpy as np

from .grid import to_grid
from .types import Task


def load_challenges(path: str | Path) -> dict[str, dict]:
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)


def load_solutions(path: str | Path) -> dict[str, list]:
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)


def tasks_from_challenges(challenges: dict[str, dict]) -> list[Task]:
    tasks = []
    for tid in sorted(challenges.keys()):
        ch = challenges[tid]
        train = [(to_grid(p["input"]), to_grid(p["output"])) for p in ch["train"]]
        test = [to_grid(p["input"]) for p in ch["test"]]
        tasks.append(Task(task_id=tid, train=train, test=test))
    return tasks


def load_tasks(
    data_dir: str | Path,
    split: str = "evaluation",
    limit: Optional[int] = None,
    task_ids: Optional[list[str]] = None,
) -> list[Task]:
    data_dir = Path(data_dir)
    challenges = load_challenges(data_dir / f"arc-agi_{split}_challenges.json")
    if task_ids:
        challenges = {k: v for k, v in challenges.items() if k in task_ids}
    tasks = tasks_from_challenges(challenges)
    if limit:
        tasks = tasks[:limit]
    return tasks


def load_solutions_for(data_dir: str | Path, split: str = "evaluation") -> Optional[dict]:
    path = Path(data_dir) / f"arc-agi_{split}_solutions.json"
    if not path.exists():
        return None
    return load_solutions(path)


In [ ]:
%%writefile /kaggle/working/h3/harness/solver.py
"""Per-task synthesis loop + full-run orchestration.

Loop shape (Poetiq-derived, but pool-preserving): Poetiq RETURNS at the first
train-verified program; we keep synthesizing to build a *verified set*, then
disambiguate offline. Rationale: many distinct programs pass all train pairs
yet disagree on the test input, and pass@2 rewards picking the right two —
that decision, not generation, is this branch's contribution.
"""
from __future__ import annotations

import time
from concurrent.futures import ThreadPoolExecutor
from dataclasses import dataclass, field
from typing import Callable, Optional, Sequence

import numpy as np

from .budget import Budget
from .disambiguate import DisambiguationConfig, FeatureBundle, disambiguate
from .extraction import pick_program
from .grid import grid_key, grids_equal
from .llm import LLMClient
from .oracle import build_feedback, evaluate_program, run_on_inputs
from .pool import ProgramPool
from .prompts import (
    build_adjudication_messages,
    build_loo_messages,
    build_solver_messages,
    build_verified_nudge_messages,
)
from .sandbox import Sandbox
from .types import Grid, Task, TaskResult


@dataclass
class SolverConfig:
    num_experts: int = 2
    max_iterations: int = 5          # per expert
    max_solutions_feedback: int = 5
    temperature: float = 1.0
    reasoning_effort: str = "low"    # NEVER 'high' (HTTP 500 on Qwen3.8)
    max_completion_tokens: int = 3072
    stop_distinct_behaviors: int = 4 # early stop: enough verified diversity
    patience_no_new: int = 3         # stop after N consecutive fruitless calls
    min_calls: int = 4               # explore at least this many calls
    sandbox_timeout: float = 2.0
    use_adjudication: bool = True
    adjudication_max_calls: int = 1
    use_loo: bool = True
    loo_max_folds: int = 3
    loo_min_pairs: int = 3           # 2-pair tasks: a fold leaves 1 pair (useless)
    seed: int = 0
    disambiguation: DisambiguationConfig = field(default_factory=DisambiguationConfig)


def _estimate_tokens(messages: list[dict]) -> int:
    return sum(len(m.get("content", "")) for m in messages) // 4


def solve_task(
    task: Task,
    llm: LLMClient,
    executor: Sandbox,
    budget: Optional[Budget],
    cfg: SolverConfig,
    time_cap_s: Optional[float] = None,
    on_bundle: Optional[Callable[[Task, ProgramPool, FeatureBundle], None]] = None,
) -> TaskResult:
    t_start = time.time()
    import zlib

    rng = np.random.default_rng(cfg.seed + zlib.crc32(task.task_id.encode()) % 10_000)
    pool = ProgramPool()
    calls = 0
    prompt_tokens = 0
    completion_tokens = 0
    no_new = 0
    last_verified_behavior_count = 0

    def _budget_ok(messages) -> bool:
        if budget is None:
            return True
        return budget.can_call(task.task_id, _estimate_tokens(messages) + cfg.max_completion_tokens)

    def _charge(resp) -> None:
        nonlocal prompt_tokens, completion_tokens
        prompt_tokens += resp.prompt_tokens
        completion_tokens += resp.completion_tokens
        if budget is not None:
            budget.charge(task.task_id, resp.prompt_tokens, resp.completion_tokens)

    def _add(code: str, source: str) -> bool:
        """Evaluate + add to pool. Returns True when behavior is new."""
        ev = evaluate_program(executor, code, task.train)
        test_outs: list[Optional[Grid]] = []
        if ev.verified:
            test_outs = run_on_inputs(executor, code, task.test)
        idx, is_new = pool.add(code, source, ev, test_outs)
        return is_new

    # ---- synthesis loop -----------------------------------------------------
    for expert in range(cfg.num_experts):
        for it in range(cfg.max_iterations):
            if budget is not None and budget.expired():
                break
            if time_cap_s is not None and time.time() - t_start > time_cap_s:
                break
            verified_groups = pool.verified_behaviors()
            n_beh = len([k for k in verified_groups if k != ("incomplete",)])
            if n_beh >= cfg.stop_distinct_behaviors and calls >= cfg.min_calls:
                break
            if calls >= cfg.min_calls and no_new >= cfg.patience_no_new:
                break

            verified = pool.verified()
            if verified:
                rep = max(verified, key=lambda i: len(pool.candidates[i].code) * -1)
                messages = build_verified_nudge_messages(
                    task, pool.candidates[rep].code, iteration=it, seed=cfg.seed + expert * 97
                )
            else:
                partials = []
                for i in pool.best_partial(n=cfg.max_solutions_feedback):
                    c = pool.candidates[i]
                    fb, _ = build_feedback(c.eval, task.train)
                    partials.append((c.code, fb, c.soft_mean))
                messages = build_solver_messages(
                    task,
                    iteration=it,
                    variant="A" if (expert + it) % 2 == 0 else "B",
                    partials=partials,
                    max_partials=cfg.max_solutions_feedback,
                    shuffle_seed=cfg.seed + expert * 97,
                )
            if not _budget_ok(messages):
                break
            try:
                resp = llm.chat(
                    messages,
                    temperature=cfg.temperature,
                    max_tokens=cfg.max_completion_tokens,
                    reasoning_effort=cfg.reasoning_effort,
                    seed=cfg.seed + expert * 1000 + it,
                )
            except Exception:
                no_new += 1
                calls += 1
                continue
            _charge(resp)
            calls += 1
            code = pick_program(resp.text)
            if code is None:
                no_new += 1
                continue
            is_new = _add(code, f"e{expert}i{it}")
            groups_now = len([k for k in pool.verified_behaviors() if k != ("incomplete",)])
            if is_new and groups_now > last_verified_behavior_count:
                no_new = 0
            elif is_new:
                no_new = max(0, no_new - 1)
            else:
                no_new += 1
            last_verified_behavior_count = groups_now
        if budget is not None and budget.expired():
            break

    # ---- no verified program: fall back to best partials (never discard) ----
    if not pool.verified():
        for i in pool.best_partial(n=5):
            c = pool.candidates[i]
            if not c.test_outputs:
                c.test_outputs = run_on_inputs(executor, c.code, task.test)

    verified_groups = {k: v for k, v in pool.verified_behaviors().items() if k != ("incomplete",)}
    disagreement = len(verified_groups) >= 2

    # ---- adjudication (gated LLM call: only on persistent disagreement) ----
    adj_votes: dict[tuple, float] = {}
    if (
        disagreement
        and cfg.use_adjudication
        and (budget is None or not budget.expired())
        and (time_cap_s is None or time.time() - t_start < time_cap_s * 0.8)
    ):
        reps = []
        for key, members in sorted(verified_groups.items(), key=lambda kv: -len(kv[1]))[:3]:
            rep = min(members, key=lambda i: len(pool.candidates[i].code))
            outs = pool.candidates[rep].test_outputs
            reps.append((pool.candidates[rep].code, outs[0] if outs else None))
        if len(reps) >= 2:
            messages = build_adjudication_messages(task, reps)
            if _budget_ok(messages):
                try:
                    resp = llm.chat(
                        messages,
                        temperature=0.4,
                        max_tokens=cfg.max_completion_tokens,
                        reasoning_effort="medium",
                        seed=cfg.seed + 7777,
                    )
                    _charge(resp)
                    code = pick_program(resp.text)
                    if code:
                        ev = evaluate_program(executor, code, task.train)
                        if ev.verified:
                            outs = run_on_inputs(executor, code, task.test)
                            pool.add(code, "adj", ev, outs)
                            if all(o is not None for o in outs):
                                key = tuple(grid_key(o) for o in outs)
                                adj_votes[key] = adj_votes.get(key, 0.0) + 1.0
                except Exception:
                    pass

    # ---- LOO corroboration (gated: disagreement + >=3 pairs + budget) -------
    loo_map: dict[tuple, float] = {}
    if (
        disagreement
        and cfg.use_loo
        and task.n_pairs >= cfg.loo_min_pairs
        and (budget is None or not budget.expired())
        and (time_cap_s is None or time.time() - t_start < time_cap_s * 0.9)
    ):
        folds = list(range(task.n_pairs))
        if len(folds) > cfg.loo_max_folds:
            folds = [int(i) for i in rng.choice(folds, size=cfg.loo_max_folds, replace=False)]
        n_pass = 0
        for fold in folds:
            messages = build_loo_messages(task, omit_pair=fold, seed=cfg.seed)
            if not _budget_ok(messages):
                break
            try:
                resp = llm.chat(
                    messages,
                    temperature=cfg.temperature,
                    max_tokens=cfg.max_completion_tokens,
                    reasoning_effort=cfg.reasoning_effort,
                    seed=cfg.seed + 31 * (fold + 1),
                )
            except Exception:
                continue
            _charge(resp)
            code = pick_program(resp.text)
            if not code:
                continue
            # held-out check: run on the withheld pair ONLY
            x_h, y_h = task.train[fold]
            res = executor.run_batch([(code, x_h)])
            if not (res[0].ok and res[0].output is not None):
                continue
            if not grids_equal(res[0].output, y_h):
                continue
            n_pass += 1
            outs = run_on_inputs(executor, code, task.test)
            pool.add(code, f"loo{fold}", evaluate_program(executor, code, task.train), outs)
            if all(o is not None for o in outs):
                key = tuple(grid_key(o) for o in outs)
                # credit the matching verified behavior group
                if key in verified_groups:
                    loo_map[key] = loo_map.get(key, 0.0) + 1.0
        if n_pass:
            for k in loo_map:
                loo_map[k] /= n_pass

    # ---- disambiguation (pure numpy + sandbox: zero LLM tokens) -------------
    fallback_outputs = [x for x in task.test]  # identity guess (writer re-fallbacks if None)
    result, attempts, bundle = disambiguate(
        task,
        pool,
        executor,
        cfg=cfg.disambiguation,
        rng=np.random.default_rng(cfg.seed + 1234),
        loo_map=loo_map or None,
        adj_votes=adj_votes or None,
        fallback_outputs=fallback_outputs,
    )
    if on_bundle is not None:
        on_bundle(task, pool, bundle)

    if budget is not None:
        budget.mark_task_done()
    return TaskResult(
        task_id=task.task_id,
        attempts=attempts,
        n_candidates=len(pool),
        n_verified=len(pool.verified()),
        n_distinct_verified_behaviors=len(verified_groups),
        llm_calls=calls,
        prompt_tokens=prompt_tokens,
        completion_tokens=completion_tokens,
        wall_s=round(time.time() - t_start, 2),
        disambiguation=result,
        extras={"adj_votes": {str(k): v for k, v in adj_votes.items()}, "loo_map": {str(k): v for k, v in loo_map.items()}},
    )


def solve_all(
    tasks: Sequence[Task],
    llm: LLMClient,
    budget: Optional[Budget],
    cfg: SolverConfig,
    submission_writer=None,
    workers: int = 1,
    executor_factory: Optional[Callable[[], Sandbox]] = None,
    on_task: Optional[Callable[[TaskResult], None]] = None,
    on_bundle: Optional[Callable[[Task, ProgramPool, FeatureBundle], None]] = None,
) -> dict:
    """Run all tasks (thread-parallel; each thread gets its own sandbox worker)."""
    results: list[TaskResult] = []
    import threading

    results_lock = threading.Lock()
    n_done = 0

    def _worker(task: Task) -> TaskResult:
        nonlocal n_done
        executor = (executor_factory() if executor_factory else Sandbox(per_job_timeout=cfg.sandbox_timeout))
        try:
            remaining = max(1, len(tasks) - n_done)
            cap = budget.per_task_time_cap(remaining) if budget is not None else None
            res = solve_task(task, llm, executor, budget, cfg, time_cap_s=cap, on_bundle=on_bundle)
        finally:
            executor.close()
        if submission_writer is not None:
            submission_writer.update(task.task_id, res.attempts)
        if on_task is not None:
            on_task(res)
        with results_lock:
            results.append(res)
            n_done += 1
        return res

    if workers <= 1:
        for task in tasks:
            if budget is not None and budget.expired():
                break
            _worker(task)
    else:
        with ThreadPoolExecutor(max_workers=workers) as ex:
            futures = []
            for task in tasks:
                if budget is not None and budget.expired():
                    break
                futures.append(ex.submit(_worker, task))
            for f in futures:
                try:
                    f.result()
                except Exception as e:  # keep going; watchdog guarantees a file
                    print(f"[solve_all] task failed: {e}")

    total_prompt = sum(r.prompt_tokens for r in results)
    total_completion = sum(r.completion_tokens for r in results)
    n_verified_tasks = sum(1 for r in results if r.n_verified > 0)
    return {
        "tasks_run": len(results),
        "tasks_with_verified": n_verified_tasks,
        "verified_rate": n_verified_tasks / max(1, len(results)),
        "llm_calls": sum(r.llm_calls for r in results),
        "prompt_tokens": total_prompt,
        "completion_tokens": total_completion,
        "wall_s": round(sum(r.wall_s for r in results), 1),
        "results": results,
    }


In [ ]:
%%writefile /kaggle/working/h3/harness/__init__.py
"""H3 branch harness: program synthesis over an exact oracle + verified-set
disambiguation for ARC-AGI-2 (Kaggle ARC Prize 2026)."""

from .budget import Budget, Watchdog
from .data import load_challenges, load_solutions, load_solutions_for, load_tasks, tasks_from_challenges
from .disambiguate import (
    DEFAULT_ENABLE,
    DEFAULT_WEIGHTS,
    DisambiguationConfig,
    FeatureBundle,
    compute_features,
    disambiguate,
    rank_and_select,
)
from .extraction import extract_code_blocks, pick_program
from .grid import format_csv, parse_grid_lenient, to_grid
from .llm import MockLLM, VLLMClient, library_responder, scripted_responder
from .oracle import evaluate_program, run_on_inputs, score_output
from .pool import ProgramPool
from .sandbox import LocalExecutor, Sandbox
from .solver import SolverConfig, solve_all, solve_task
from .submission import SubmissionWriter, score_submission, validate_submission
from .types import Candidate, Grid, LLMResponse, Task, TaskResult

__all__ = [
    "Budget",
    "Watchdog",
    "Candidate",
    "DEFAULT_ENABLE",
    "DEFAULT_WEIGHTS",
    "DisambiguationConfig",
    "FeatureBundle",
    "Grid",
    "LLMResponse",
    "LocalExecutor",
    "MockLLM",
    "ProgramPool",
    "Sandbox",
    "SolverConfig",
    "SubmissionWriter",
    "Task",
    "TaskResult",
    "VLLMClient",
    "compute_features",
    "disambiguate",
    "evaluate_program",
    "extract_code_blocks",
    "format_csv",
    "library_responder",
    "load_challenges",
    "load_solutions",
    "load_solutions_for",
    "load_tasks",
    "parse_grid_lenient",
    "pick_program",
    "rank_and_select",
    "run_on_inputs",
    "score_output",
    "score_submission",
    "scripted_responder",
    "solve_all",
    "solve_task",
    "tasks_from_challenges",
    "to_grid",
    "validate_submission",
]


In [ ]:
sys.path.insert(0, H3)
import harness
from harness.sandbox import Sandbox
from harness.llm import VLLMClient, MockLLM, library_responder

# sandbox smoke test (subprocess isolation + numpy worker)
_sb = Sandbox(per_job_timeout=2.0)
_r = _sb.run("import numpy as np\n\ndef transform(grid):\n    return grid + 1\n", [[0, 1], [2, 3]])
assert _r.ok, _r.error
_sb.close()
print("harness import + sandbox OK:", harness.__name__)


## Launch vLLM (offline, TP=4) or fall back to the CPU template MockLLM

Verified launch line for Qwen3.8 (traps handled): `--reasoning-parser qwen3`,
`--tool-call-parser qwen3_coder`, `--enable-prefix-caching` with `--mamba-cache-mode align`
(hybrid Gated DeltaNet), NO speculative decoding (breaks constrained decoding), and the client
pins `reasoning_effort` to low/medium — `high` raises HTTP 500 (QwenLM/Qwen3.8#217) and the
default `xhigh` would emit 8-16K thinking tokens per call (~4x over budget).

In [ ]:
import json as _json, time as _time, urllib.request, urllib.error

VLLM_URL = "http://127.0.0.1:8000/v1"
MODEL_NAME = "qwen3.8-27b"
LLM = None
_vllm_proc = None

def _server_ready(timeout_s=60):
    t0 = _time.time()
    while _time.time() - t0 < timeout_s:
        try:
            with urllib.request.urlopen(VLLM_URL + "/models", timeout=5) as r:
                if r.status == 200:
                    return True
        except Exception:
            _time.sleep(5)
    return False

def _launch_vllm(max_model_len):
    cmd = [
        sys.executable, "-m", "vllm.entrypoints.openai.api_server",
        "--model", MODEL_DIR,
        "--served-model-name", MODEL_NAME,
        "--tensor-parallel-size", "4",
        "--max-model-len", str(max_model_len),
        "--gpu-memory-utilization", "0.90",
        "--reasoning-parser", "qwen3",
        "--enable-auto-tool-choice", "--tool-call-parser", "qwen3_coder",
        "--enable-prefix-caching", "--mamba-cache-mode", "align",
        "--port", "8000",
    ]
    logf = open(os.path.join(WORKING, "vllm.log"), "wb")
    return subprocess.Popen(cmd, stdout=logf, stderr=subprocess.STDOUT)

if HAS_GPU and MODEL_DIR is not None:
    _vllm_proc = _launch_vllm(262144)
    if not _server_ready(timeout_s=1800):
        # KV/activation headroom issue at 256K context -> retry smaller
        try:
            _vllm_proc.kill()
        except Exception:
            pass
        print("first launch failed; retrying with --max-model-len 32768 (see vllm.log)")
        _vllm_proc = _launch_vllm(32768)
        if not _server_ready(timeout_s=1800):
            raise RuntimeError("vLLM failed to become ready; inspect /kaggle/working/vllm.log")
    LLM = VLLMClient(base_url=VLLM_URL, model=MODEL_NAME, request_timeout=1200.0, max_retries=3)
    _probe = LLM.chat(
        [{"role": "user", "content": "Reply with exactly: OK"}],
        temperature=0.0, max_tokens=512, reasoning_effort="low",
    )
    print("vLLM smoke reply:", _probe.text[:120].replace("\n", " "))
else:
    print("No GPU/model available -> CPU fallback mode (template MockLLM).")
    print("The notebook still produces a schema-valid submission via the oracle fallback path.")
    LLM = MockLLM(library_responder())


In [ ]:
from harness.budget import Budget, Watchdog
from harness.data import load_challenges, tasks_from_challenges
from harness.disambiguate import DisambiguationConfig
from harness.solver import SolverConfig, solve_all
from harness.submission import SubmissionWriter, validate_submission

assert DATA_DIR is not None, "competition data not attached"
test_file = "arc-agi_test_challenges.json" if RERUN else "arc-agi_evaluation_challenges.json"
challenges = load_challenges(os.path.join(DATA_DIR, test_file))
tasks = tasks_from_challenges(challenges)
print(f"loaded {len(tasks)} tasks from {test_file}")

SUB_PATH = os.path.join(WORKING, "submission.json")
writer = SubmissionWriter(SUB_PATH, tasks)   # writes all-task fallback immediately

budget = Budget(
    deadline_ts=END_TS - 300,          # stop starting tasks 5 min before the buffer edge
    per_task_tokens=75_000,            # HYPOTHESIS.md budget: ~57K raw prompt + ~13.5K completion
    total_tokens=24_000_000,
    reserve_seconds=0.0,
)

def _finalize():
    try:
        writer.flush()
        with open(SUB_PATH) as f:
            data = json.load(f)
        problems = validate_submission(data, tasks)
        print("[watchdog] final submission valid:", not problems)
    except Exception as e:
        print("[watchdog] finalize error:", e)

watchdog = Watchdog(END_TS - 120, _finalize, hard_kill_grace_s=120)
watchdog.start()

cfg = SolverConfig(
    num_experts=2,
    max_iterations=5,
    stop_distinct_behaviors=4,
    patience_no_new=3,
    sandbox_timeout=3.0,
    use_adjudication=True,
    use_loo=True,
    loo_max_folds=3,
    seed=20260326,
    disambiguation=DisambiguationConfig(seed=20260326),
)

_t0 = time.time()
summary = solve_all(
    tasks, LLM, budget, cfg,
    submission_writer=writer,
    workers=4,
    executor_factory=lambda: Sandbox(per_job_timeout=3.0, startup_timeout=60.0),
    on_task=lambda r: print(f"  [{r.task_id}] cands={r.n_candidates} verified={r.n_verified} "
                            f"behaviors={r.n_distinct_verified_behaviors} calls={r.llm_calls} "
                            f"tok={r.prompt_tokens + r.completion_tokens} wall={r.wall_s:.0f}s", flush=True),
)
watchdog.cancel()
_finalize()
print("solve_all done in", round(time.time() - _t0, 1), "s")
print({k: v for k, v in summary.items() if k != "results"})
print("budget:", budget.snapshot())


In [ ]:
import json as _json

with open(SUB_PATH) as f:
    _data = _json.load(f)
_problems = validate_submission(_data, tasks)
print("submission path:", SUB_PATH)
print("task_ids present:", len(_data), "of", len(tasks))
print("schema problems:", _problems[:5] if _problems else "NONE - valid")
_n_att = sum(len(v) for v in _data.values())
print("attempt packs:", _n_att)
assert not _problems, "submission schema invalid"
